# CNN+LSTM DDoS Detector — Live Testing & Evaluation

This notebook tests the trained CNN+LSTM model on **live simulated data** from the SDN monitoring API.

**Sections:**
1. Imports & Config
2. Load Model & Artifacts
3. Monitoring API *(paste your code here)*
4. Live Preprocessing Pipeline
5. Real-Time Prediction Loop
6. Evaluation & Metrics
7. Time-Series Visualization

## 1. Imports & Configuration

In [7]:
# ═══════════════════════════════════════════════════════════════════
# Section 1 — Imports & Configuration
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('TkAgg')  # or 'Agg' for headless
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, sys, time, pickle, re, warnings
from pathlib import Path
from collections import deque
from datetime import datetime, timezone

import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    roc_curve, precision_recall_curve
)
from IPython.display import clear_output

warnings.filterwarnings('ignore')
%matplotlib notebook
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# ── Paths ──────────────────────────────────────────────────────────
ARTIFACT_DIR = 'outputs_detector'
MODEL_PATH   = os.path.join(ARTIFACT_DIR, 'model_final.keras')
SCALER_PATH  = os.path.join(ARTIFACT_DIR, 'scaler.pkl')
IMPUTER_PATH = os.path.join(ARTIFACT_DIR, 'imputer.pkl')
CONFIG_PATH  = os.path.join(ARTIFACT_DIR, 'config.json')
CALIB_PATH   = os.path.join(ARTIFACT_DIR, 'calibration.json')

# ── Live Testing Config ────────────────────────────────────────────
PREDICTION_INTERVAL = 2        # seconds between predictions
LIVE_HISTORY_MAX    = 5000     # max samples to keep in memory
PLOT_WINDOW         = 200      # samples to show in real-time plot

print('✅ Imports loaded')

✅ Imports loaded


## 2. Load Model, Scaler, Imputer & Config

In [8]:
# ═══════════════════════════════════════════════════════════════════
# Section 2 — Load Trained Artifacts
# ═══════════════════════════════════════════════════════════════════

# Config
with open(CONFIG_PATH) as f:
    config = json.load(f)

FEATURE_COLS    = config['feature_cols']       # ordered list of 135 features
WINDOW_LEN      = config['BEST_WINDOW_LEN']    # 5
BEST_THRESHOLD  = config['BEST_THRESHOLD']      # 0.847
N_FEATURES      = config['n_features']           # 135

# Calibration
with open(CALIB_PATH) as f:
    calib = json.load(f)
TEMPERATURE = calib['temperature']               # 0.957

# Model
model = keras.models.load_model(MODEL_PATH)
print(f'✅ Model loaded: {MODEL_PATH}')
model.summary()

# Scaler & Imputer
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)
with open(IMPUTER_PATH, 'rb') as f:
    imputer = pickle.load(f)

print(f'\n✅ Config loaded:')
print(f'  Features:  {N_FEATURES}')
print(f'  Window:    {WINDOW_LEN}')
print(f'  Threshold: {BEST_THRESHOLD:.4f}')
print(f'  Temperature: {TEMPERATURE:.4f}')
print(f'  Feature list (first 10): {FEATURE_COLS[:10]}')

✅ Model loaded: outputs_detector/model_final.keras


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5, 135)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 5, 48)          │        19,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 5, 48)          │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 5, 48)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 5, 48)          │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 5, 48)          │         6,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 5, 48)          │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 5, 48)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 5, 48)          │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        28,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 173,237 (676.71 KB)

 Trainable params: 57,681 (225.32 KB)

 Non-trainable params: 192 (768.00 B)

 Optimizer params: 115,364 (450.64 KB)


✅ Config loaded:
  Features:  135
  Window:    5
  Threshold: 0.8472
  Temperature: 0.9568
  Feature list (first 10): ['onos1_flows', 'onos1_packets', 'onos1_bytes', 'onos1_flows_rate', 'onos1_packets_rate', 'onos1_bytes_rate', 'onos1_cpu', 'onos1_mem', 'onos1_pktin_total', 'onos1_pktin_total_rate']


## 3. Monitoring API

**Paste your SDN Controller-Level Monitoring API code in the cell below.**  
It should define: `start_monitoring()`, `stop_monitoring()`, `set_label()`, `show_status()`, `collected_samples`, `training_queue`, etc.

## 4. Live Preprocessing Pipeline

This replicates the **exact** preprocessing from the training notebook:
- Feature mapping (`cluster_syn_flows` → `syn_flows`, etc.)
- Memory string conversion (GiB → bytes)
- Median imputation → StandardScaler
- Sliding window creation

In [9]:
# ═══════════════════════════════════════════════════════════════════
# Section 4 — Live Preprocessing Pipeline
# ═══════════════════════════════════════════════════════════════════

def parse_memory_string(val):
    """Convert memory strings like '1.638GiB' to numeric bytes."""
    if isinstance(val, (int, float)):
        return float(val)
    if not isinstance(val, str):
        return np.nan
    val = val.strip()
    multipliers = {
        'gib': 1024**3, 'gb': 1e9,
        'mib': 1024**2, 'mb': 1e6,
        'kib': 1024,    'kb': 1e3,
        'b': 1
    }
    for suffix, mult in multipliers.items():
        if val.lower().endswith(suffix):
            try:
                return float(val[:len(val)-len(suffix)]) * mult
            except ValueError:
                return np.nan
    try:
        return float(val)
    except ValueError:
        return np.nan


def apply_feature_mapping(df):
    """
    Replicate EXACTLY the same feature engineering from training:
    - Keep cluster_syn_flows values as syn_flows
    - Keep cluster_syn_ratio values as syn_ratio
    - Calculate well_known_port_flows proxy
    - Calculate five_unique
    """
    # ── cluster_syn_flows → syn_flows (keep cluster version) ──
    if 'cluster_syn_flows' in df.columns and 'syn_flows' in df.columns:
        df.drop(columns=['syn_flows'], inplace=True)
        df.rename(columns={'cluster_syn_flows': 'syn_flows'}, inplace=True)
    elif 'cluster_syn_flows' in df.columns:
        df.rename(columns={'cluster_syn_flows': 'syn_flows'}, inplace=True)

    # ── cluster_syn_ratio → syn_ratio (keep cluster version) ──
    if 'cluster_syn_ratio' in df.columns and 'syn_ratio' in df.columns:
        df.drop(columns=['syn_ratio'], inplace=True)
        df.rename(columns={'cluster_syn_ratio': 'syn_ratio'}, inplace=True)
    elif 'cluster_syn_ratio' in df.columns:
        df.rename(columns={'cluster_syn_ratio': 'syn_ratio'}, inplace=True)

    # ── well_known_port_flows (proxy) ──
    if 'unique_dst_ports' in df.columns:
        udp = df['unique_dst_ports'].fillna(0).clip(lower=0)
        if 'well_known_port_flows' not in df.columns or df['well_known_port_flows'].isna().all():
            df['well_known_port_flows'] = np.where(
                udp > 0,
                np.minimum(udp, 1024) / np.maximum(udp, 1),
                0.0
            )

    # ── five_unique ──
    if 'unique_srcs' in df.columns and 'unique_dsts' in df.columns:
        if 'five_unique' not in df.columns or df['five_unique'].isna().all():
            df['five_unique'] = df['unique_srcs'].fillna(0) + df['unique_dsts'].fillna(0)

    return df


def preprocess_live_sample(raw_df):
    """
    Full preprocessing pipeline for live data.
    Input:  raw DataFrame from monitoring API (any number of rows)
    Output: preprocessed feature array ready for windowing
    """
    df = raw_df.copy()

    # 1. Feature mapping
    df = apply_feature_mapping(df)

    # 2. Convert memory strings to bytes
    mem_cols = [c for c in df.columns if '_mem' in c]
    for col in mem_cols:
        df[col] = df[col].apply(parse_memory_string)

    # 3. Replace inf with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 4. Ensure all required features exist
    for col in FEATURE_COLS:
        if col not in df.columns:
            df[col] = np.nan

    # 5. Select features in EXACT training order
    X = df[FEATURE_COLS].values.astype(np.float64)

    # 6. Impute (using training-fitted imputer)
    X = imputer.transform(X)

    # 7. Scale (using training-fitted scaler)
    X = scaler.transform(X)

    return X


def create_windows(X, window_len):
    """
    Create sliding windows from preprocessed feature array.
    Returns windows of shape (n_windows, window_len, n_features)
    """
    if len(X) < window_len:
        return np.empty((0, window_len, X.shape[1]))
    windows = []
    for i in range(len(X) - window_len + 1):
        windows.append(X[i:i + window_len])
    return np.array(windows)


def calibrate_probability(raw_prob, temperature):
    """Apply temperature scaling to raw sigmoid output."""
    eps = 1e-7
    logits = np.log(np.clip(raw_prob, eps, 1 - eps) /
                    np.clip(1 - raw_prob, eps, 1 - eps))
    return 1.0 / (1.0 + np.exp(-logits / temperature))


print('✅ Preprocessing pipeline defined.')
print(f'   Features expected: {N_FEATURES}')
print(f'   Window length:     {WINDOW_LEN}')

✅ Preprocessing pipeline defined.
   Features expected: 135
   Window length:     5


## 5. Start Monitoring & Real-Time Prediction

In [9]:
# ═══════════════════════════════════════════════════════════════════
# Section 5a — Start Monitoring (benign traffic first)
# ═══════════════════════════════════════════════════════════════════

start_monitoring(label=0)
print(f'Monitoring started. Waiting for initial {WINDOW_LEN} samples...')
time.sleep(WINDOW_LEN * 2 + 2)  # wait for enough samples
show_status()

[RESET] prev_* cleared. reason=start_monitoring
✓ Monitoring started (label=0). Sampling every 2s
Monitoring started. Waiting for initial 5 samples...
MONITORING STATUS (schema v1.3.3)
Active: True
Collected samples (in memory): 573
Training queue size: 573

------------------------------------------------------------
LAST SAMPLE
------------------------------------------------------------
Timestamp: 2026-02-27T08:20:03.909697+00:00
Sample ID: 573 | Label: 0 | Warmup: 0
Total flows: 4 | Total packets: 2061


In [24]:
# ═══════════════════════════════════════════════════════════════════
# Section 5b — Real-Time Prediction Loop
# ═══════════════════════════════════════════════════════════════════
# This cell runs continuously. Press the Stop button to end.

# Storage for predictions
prediction_history = []

# Rolling buffer for windowing (keeps last WINDOW_LEN preprocessed rows)
feature_buffer = deque(maxlen=LIVE_HISTORY_MAX)

# Set up live plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('CNN+LSTM DDoS Detector — Live Predictions', fontsize=14, fontweight='bold')

print('🔴 LIVE PREDICTION LOOP STARTED')
print('   Press Stop (■) to end the loop.\n')

sample_counter = 0
last_processed = 0

try:
    while True:
        # Wait for new samples
        time.sleep(PREDICTION_INTERVAL)

        with data_lock:
            current_samples = list(collected_samples)

        n_available = len(current_samples)
        if n_available <= last_processed:
            continue

        # Process only NEW samples
        new_samples = current_samples[last_processed:]
        last_processed = n_available

        # Convert to DataFrame and preprocess
        df_new = pd.DataFrame(new_samples)
        X_new = preprocess_live_sample(df_new)

        # Add to rolling buffer
        for row in X_new:
            feature_buffer.append(row)

        # Need at least WINDOW_LEN samples in buffer
        if len(feature_buffer) < WINDOW_LEN:
            print(f'  Buffering... {len(feature_buffer)}/{WINDOW_LEN}')
            continue

        # Create window from the LAST WINDOW_LEN samples
        buffer_arr = np.array(list(feature_buffer))
        window = buffer_arr[-WINDOW_LEN:].reshape(1, WINDOW_LEN, N_FEATURES)

        # Predict
        raw_prob = model.predict(window, verbose=0).ravel()[0]
        cal_prob = calibrate_probability(raw_prob, TEMPERATURE)
        binary   = int(cal_prob >= BEST_THRESHOLD)
        confidence = abs(cal_prob - 0.5) * 2

        # Get ground truth label from latest sample
        true_label = new_samples[-1].get('label', -1)
        ts_utc     = new_samples[-1].get('timestamp_utc', '')
        sample_id  = new_samples[-1].get('sample_id', sample_counter)

        # Severity score
        last_row = new_samples[-1]
        syn_r = float(last_row.get('syn_ratio', 0) or 0)
        syn_f = float(last_row.get('syn_flows', 0) or 0)
        far   = float(last_row.get('flow_arrival_rate', 0) or 0)
        udst  = float(last_row.get('unique_dsts', 1) or 1)

        prediction_history.append({
            'sample_id':    sample_id,
            'timestamp':    ts_utc,
            'true_label':   true_label,
            'raw_prob':     raw_prob,
            'cal_prob':     cal_prob,
            'prediction':   binary,
            'confidence':   confidence,
            'syn_ratio':    syn_r,
            'syn_flows':    syn_f,
            'flow_arrival_rate': far,
            'unique_dsts':  udst,
            'total_flows':  last_row.get('total_flows', 0),
        })

        sample_counter += 1

        # Print status
        status = '🔴 ATTACK' if binary == 1 else '🟢 BENIGN'
        match  = '✅' if binary == true_label else '❌'
        print(f'[{sample_id:>5}] {status}  prob={cal_prob:.4f}  '
              f'conf={confidence:.3f}  true={true_label}  {match}')

        # ── Update live plot every 5 predictions ────────────────────
        if sample_counter % 5 == 0 and len(prediction_history) > 1:
            df_hist = pd.DataFrame(prediction_history[-PLOT_WINDOW:])

            for ax in axes:
                ax.clear()

            # Plot 1: Probability vs Threshold
            axes[0].plot(df_hist.index, df_hist['cal_prob'], 'b-', linewidth=1, label='P(attack)')
            axes[0].axhline(y=BEST_THRESHOLD, color='r', linestyle='--', alpha=0.7, label=f'Threshold={BEST_THRESHOLD:.3f}')
            axes[0].fill_between(df_hist.index, 0, 1,
                                 where=df_hist['true_label'] == 1,
                                 alpha=0.15, color='red', label='True attack')
            axes[0].set_ylabel('Attack Probability')
            axes[0].set_ylim(-0.05, 1.05)
            axes[0].legend(loc='upper right', fontsize=8)
            axes[0].set_title('Live Attack Probability', fontsize=11)

            # Plot 2: Binary Prediction vs True Label
            axes[1].step(df_hist.index, df_hist['prediction'], 'r-', linewidth=1.5, where='post', label='Predicted')
            axes[1].step(df_hist.index, df_hist['true_label'], 'g--', linewidth=1.5, where='post', label='True label')
            axes[1].set_ylabel('Label')
            axes[1].set_yticks([0, 1])
            axes[1].set_yticklabels(['Benign', 'Attack'])
            axes[1].legend(loc='upper right', fontsize=8)
            axes[1].set_title('Prediction vs Ground Truth', fontsize=11)

            # Plot 3: Total Flows (traffic volume)
            axes[2].plot(df_hist.index, df_hist['total_flows'], 'purple', linewidth=1)
            axes[2].fill_between(df_hist.index, 0, df_hist['total_flows'], alpha=0.3, color='purple')
            axes[2].set_ylabel('Total Flows')
            axes[2].set_xlabel('Sample')
            axes[2].set_title('Network Traffic Volume', fontsize=11)

            fig.tight_layout()
            fig.canvas.draw()
            fig.canvas.flush_events()

except KeyboardInterrupt:
    print(f'\n\n🛑 Prediction loop stopped after {sample_counter} predictions.')

<IPython.core.display.Javascript object>

🔴 LIVE PREDICTION LOOP STARTED
   Press Stop (■) to end the loop.

[   99] 🟢 BENIGN  prob=0.0704  conf=0.859  true=0  ✅
[  101] 🟢 BENIGN  prob=0.0204  conf=0.959  true=0  ✅
[  102] 🟢 BENIGN  prob=0.0416  conf=0.917  true=0  ✅
[  103] 🟢 BENIGN  prob=0.0103  conf=0.979  true=0  ✅
[  104] 🟢 BENIGN  prob=0.0075  conf=0.985  true=0  ✅
[  105] 🟢 BENIGN  prob=0.0084  conf=0.983  true=0  ✅
[  106] 🟢 BENIGN  prob=0.0019  conf=0.996  true=0  ✅
[  107] 🟢 BENIGN  prob=0.0031  conf=0.994  true=0  ✅
[  108] 🟢 BENIGN  prob=0.0045  conf=0.991  true=0  ✅
[  109] 🔴 ATTACK  prob=0.9996  conf=0.999  true=1  ✅
[  110] 🔴 ATTACK  prob=0.9696  conf=0.939  true=1  ✅
[  111] 🔴 ATTACK  prob=0.9965  conf=0.993  true=1  ✅
[  112] 🔴 ATTACK  prob=0.9992  conf=0.998  true=1  ✅
[  113] 🔴 ATTACK  prob=0.9995  conf=0.999  true=1  ✅
[  114] 🔴 ATTACK  prob=0.9991  conf=0.998  true=1  ✅
[  115] 🔴 ATTACK  prob=0.9992  conf=0.998  true=1  ✅
[  116] 🔴 ATTACK  prob=0.9991  conf=0.998  true=1  ✅
[  117] 🔴 ATTACK  prob=0.9981  c

## 5c. Label Control

Run the cell below to **switch labels** during the experiment.  
- `set_label(0)` → mark as **benign** traffic
- `set_label(1)` → mark as **attack** traffic

In [ ]:
# ── Switch label here when you start/stop attacks ──
set_label(1)   # Change to 0 for benign, 1 for attack

In [10]:
# ── Stop monitoring when done ──
stop_monitoring()
print(f'Total predictions collected: {len(prediction_history)}')

✓ Monitoring stopped. 613 samples saved to: Online_test_1.csv
Total predictions collected: 578


## 6. Post-Experiment Evaluation

In [11]:
# ═══════════════════════════════════════════════════════════════════
# Section 6 — Comprehensive Evaluation
# ═══════════════════════════════════════════════════════════════════

df_results = pd.DataFrame(prediction_history)

# Filter out samples with unknown labels
df_eval = df_results[df_results['true_label'].isin([0, 1])].copy()
print(f'Evaluating {len(df_eval)} samples (excluded {len(df_results) - len(df_eval)} with unknown labels)\n')

y_true = df_eval['true_label'].values
y_pred = df_eval['prediction'].values
y_prob = df_eval['cal_prob'].values

# ── Classification Report ──
print('═' * 60)
print('CLASSIFICATION REPORT')
print('═' * 60)
print(classification_report(y_true, y_pred, target_names=['Benign', 'Attack']))

# ── Key Metrics ──
roc_auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan')
pr_auc  = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan')
f1      = f1_score(y_true, y_pred)
prec    = precision_score(y_true, y_pred, zero_division=0)
rec     = recall_score(y_true, y_pred, zero_division=0)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
fpr = fp / max(1, fp + tn)
fnr = fn / max(1, fn + tp)

print(f'\n── Live Test Metrics ──')
print(f'  ROC-AUC     : {roc_auc:.4f}')
print(f'  PR-AUC      : {pr_auc:.4f}')
print(f'  F1          : {f1:.4f}')
print(f'  Precision   : {prec:.4f}')
print(f'  Recall      : {rec:.4f}')
print(f'  FPR         : {fpr:.4f}')
print(f'  FNR         : {fnr:.4f}')
print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')

# ── Detection Latency ──
# How many samples after an attack starts before the model detects it?
attack_starts = []
in_attack = False
for i, row in df_eval.iterrows():
    if row['true_label'] == 1 and not in_attack:
        attack_starts.append(i)
        in_attack = True
    elif row['true_label'] == 0:
        in_attack = False

latencies = []
for start_idx in attack_starts:
    subset = df_eval.loc[start_idx:]
    for j, (idx, row) in enumerate(subset.iterrows()):
        if row['true_label'] != 1:
            break
        if row['prediction'] == 1:
            latencies.append(j)
            break

if latencies:
    print(f'\n── Detection Latency ──')
    print(f'  Mean: {np.mean(latencies):.1f} samples ({np.mean(latencies) * PREDICTION_INTERVAL:.1f}s)')
    print(f'  Max:  {np.max(latencies)} samples ({np.max(latencies) * PREDICTION_INTERVAL:.1f}s)')
    print(f'  Min:  {np.min(latencies)} samples ({np.min(latencies) * PREDICTION_INTERVAL:.1f}s)')
else:
    print('\n  ℹ️  No attack episodes detected for latency measurement.')

Evaluating 578 samples (excluded 0 with unknown labels)

════════════════════════════════════════════════════════════
CLASSIFICATION REPORT
════════════════════════════════════════════════════════════
              precision    recall  f1-score   support

      Benign       0.98      0.95      0.97       335
      Attack       0.94      0.97      0.95       243

    accuracy                           0.96       578
   macro avg       0.96      0.96      0.96       578
weighted avg       0.96      0.96      0.96       578


── Live Test Metrics ──
  ROC-AUC     : 0.9924
  PR-AUC      : 0.9893
  F1          : 0.9535
  Precision   : 0.9365
  Recall      : 0.9712
  FPR         : 0.0478
  FNR         : 0.0288
  TP=236  FP=16  TN=319  FN=7

── Detection Latency ──
  Mean: 0.0 samples (0.0s)
  Max:  0 samples (0.0s)
  Min:  0 samples (0.0s)


In [12]:
# ═══════════════════════════════════════════════════════════════════
# Section 6b — Confusion Matrix Plot
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Attack'],
            yticklabels=['Benign', 'Attack'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title(f'Confusion Matrix (Live Test)\nF1={f1:.4f}')

# ROC Curve
if len(np.unique(y_true)) > 1:
    fpr_curve, tpr_curve, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr_curve, tpr_curve, 'b-', linewidth=2)
    axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'ROC Curve (AUC={roc_auc:.4f})')
    axes[1].grid(True, alpha=0.3)

# PR Curve
if len(np.unique(y_true)) > 1:
    prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_prob)
    axes[2].plot(rec_curve, prec_curve, 'g-', linewidth=2)
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title(f'PR Curve (AUC={pr_auc:.4f})')
    axes[2].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, 'live_test_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {ARTIFACT_DIR}/live_test_metrics.png')

<IPython.core.display.Javascript object>

✅ Saved: outputs_detector/live_test_metrics.png


## 7. Time-Series Visualization

In [13]:
# ═══════════════════════════════════════════════════════════════════
# Section 7 — Full Time-Series Plot
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle('CNN+LSTM Live Detection — Full Session', fontsize=14, fontweight='bold')

x = range(len(df_eval))

# 1. Attack Probability
axes[0].plot(x, df_eval['cal_prob'], 'b-', linewidth=0.8, alpha=0.9)
axes[0].axhline(y=BEST_THRESHOLD, color='r', linestyle='--', alpha=0.7,
                label=f'Threshold={BEST_THRESHOLD:.3f}')
axes[0].fill_between(x, 0, 1,
                     where=df_eval['true_label'].values == 1,
                     alpha=0.15, color='red', label='True attack window')
axes[0].set_ylabel('P(attack)')
axes[0].set_ylim(-0.05, 1.05)
axes[0].legend(fontsize=8)
axes[0].set_title('Calibrated Attack Probability')

# 2. Binary prediction vs truth
correct = (df_eval['prediction'].values == df_eval['true_label'].values).astype(int)
colors = ['green' if c else 'red' for c in correct]
axes[1].bar(x, df_eval['prediction'], color=colors, alpha=0.6, width=1.0)
axes[1].step(x, df_eval['true_label'], 'k-', linewidth=1.5, where='mid', label='True')
axes[1].set_ylabel('Label')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Benign', 'Attack'])
axes[1].legend(fontsize=8)
axes[1].set_title('Predictions (green=correct, red=wrong)')

# 3. Confidence
axes[2].plot(x, df_eval['confidence'], 'orange', linewidth=0.8)
axes[2].fill_between(x, 0, df_eval['confidence'], alpha=0.3, color='orange')
axes[2].set_ylabel('Confidence')
axes[2].set_ylim(-0.05, 1.05)
axes[2].set_title('Prediction Confidence')

# 4. Traffic features
axes[3].plot(x, df_eval['total_flows'], 'purple', linewidth=0.8, label='Total flows')
ax3b = axes[3].twinx()
ax3b.plot(x, df_eval['syn_ratio'], 'red', linewidth=0.8, alpha=0.6, label='SYN ratio')
ax3b.set_ylabel('SYN Ratio', color='red')
axes[3].set_ylabel('Total Flows', color='purple')
axes[3].set_xlabel('Sample Index')
axes[3].set_title('Network Features')
axes[3].legend(loc='upper left', fontsize=8)
ax3b.legend(loc='upper right', fontsize=8)

fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, 'live_test_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {ARTIFACT_DIR}/live_test_timeseries.png')

<IPython.core.display.Javascript object>

✅ Saved: outputs_detector/live_test_timeseries.png


In [14]:
# ═══════════════════════════════════════════════════════════════════
# Section 7b — Prediction Distribution
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram by class
benign_probs = df_eval[df_eval['true_label'] == 0]['cal_prob']
attack_probs = df_eval[df_eval['true_label'] == 1]['cal_prob']

axes[0].hist(benign_probs, bins=50, alpha=0.6, color='green', label='Benign', density=True)
axes[0].hist(attack_probs, bins=50, alpha=0.6, color='red', label='Attack', density=True)
axes[0].axvline(x=BEST_THRESHOLD, color='black', linestyle='--', label=f'Threshold={BEST_THRESHOLD:.3f}')
axes[0].set_xlabel('Calibrated Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Prediction Distribution by True Class')
axes[0].legend()

# Confidence by correctness
correct_conf = df_eval[df_eval['prediction'] == df_eval['true_label']]['confidence']
wrong_conf   = df_eval[df_eval['prediction'] != df_eval['true_label']]['confidence']

axes[1].hist(correct_conf, bins=50, alpha=0.6, color='green', label=f'Correct (n={len(correct_conf)})', density=True)
if len(wrong_conf) > 0:
    axes[1].hist(wrong_conf, bins=50, alpha=0.6, color='red', label=f'Wrong (n={len(wrong_conf)})', density=True)
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Density')
axes[1].set_title('Confidence Distribution by Correctness')
axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, 'live_test_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {ARTIFACT_DIR}/live_test_distributions.png')

<IPython.core.display.Javascript object>

✅ Saved: outputs_detector/live_test_distributions.png


In [15]:
# ═══════════════════════════════════════════════════════════════════
# Section 7c — Save Results CSV
# ═══════════════════════════════════════════════════════════════════

results_path = os.path.join(ARTIFACT_DIR, 'live_test_results.csv')
df_results.to_csv(results_path, index=False)

# Save summary metrics
live_metrics = {
    'total_samples': len(df_eval),
    'ROC_AUC': float(roc_auc),
    'PR_AUC': float(pr_auc),
    'F1': float(f1),
    'Precision': float(prec),
    'Recall': float(rec),
    'FPR': float(fpr),
    'FNR': float(fnr),
    'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
    'threshold': BEST_THRESHOLD,
    'temperature': TEMPERATURE,
    'window_len': WINDOW_LEN,
    'timestamp': datetime.now().isoformat(),
}
with open(os.path.join(ARTIFACT_DIR, 'live_test_metrics.json'), 'w') as f:
    json.dump(live_metrics, f, indent=2)

print(f'✅ Results saved: {results_path}')
print(f'✅ Metrics saved: {ARTIFACT_DIR}/live_test_metrics.json')
print(f'\n🏁 Live testing complete!')

✅ Results saved: outputs_detector/live_test_results.csv
✅ Metrics saved: outputs_detector/live_test_metrics.json

🏁 Live testing complete!


In [7]:
stop_monitoring()

✓ Monitoring stopped. 255 samples saved to: Online_test.csv


In [37]:
"""
Full SDN Controller-Level Monitoring (LSTM → DRL ready, updated v1.4.0)
CHANGES from v1.3.3:
- ADDED per-source-IP features (Approach 1) for DRL attacker identification
- Top-6 sources ranked by packet count with 10 features each (60 new cols)
- 3 global source diversity metrics (active_sources, source_entropy, top_source_dominance)
- All existing 162 columns UNCHANGED — LSTM works without modification
"""

import requests, time, pandas as pd, numpy as np, subprocess, re, json, threading, hashlib, math, os, queue
from requests.auth import HTTPBasicAuth
from collections import defaultdict, deque
from datetime import datetime, timezone
from math import log2

# ===== Config (edit) =====
CONTROLLERS = [
    {"id": "onos1", "ip": "175.24.1.5", "port": 8181},
    {"id": "onos2", "ip": "175.24.1.6", "port": 8181},
    {"id": "onos3", "ip": "175.24.1.7", "port": 8181},
]
SWITCH_TO_CONTROLLER = {"s1":"175.24.1.7","s2":"175.24.1.6","s3":"175.24.1.6","s4":"175.24.1.5"}
AUTH = HTTPBasicAuth("onos", "rocks")
INTERVAL = 2
SWITCHES = ["s1","s2","s3","s4"]
CSV_PATH = "DRL_test_1.csv"
PARQUET_PATH = "DRL_test_1.parquet"
PARQUET_SYNC_EVERY = 100

SCHEMA_VERSION = "1.5.0"

# ---- Smoothing / evaluation knobs ----
REST_LAT_WINDOW = 2
WARMUP_SAMPLES = 0

# ---- Per-source-IP features (Approach 1) ----
TOP_K_SOURCES = 6   # track top-6 sources by packet count

# Host-to-IP mapping (must match auto_attack_cycle.sh)
HOST_TO_IP = {"h2": "10.10.0.2", "h4": "10.10.0.4", "iot2": "10.10.0.102"}

# ===== Globals =====
monitoring_active = False
monitoring_thread = None
prev_stats = {}
prev_metrics = {}
prev_flows_set = set()
collected_samples = []
parquet_pending_counter = 0
training_queue = queue.Queue()
current_label = 0
current_attacker_ip = "none"
current_attacker_host = "none"
flow_database = {}
attack_markers = []
monotonic_start = time.monotonic()

rest_lat_hist = defaultdict(lambda: deque(maxlen=REST_LAT_WINDOW))

data_lock = threading.Lock()

# ===== Helpers =====
def now_utc_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

def query(ctrl, endpoint, timeout=3.0):
    url = f"http://{ctrl['ip']}:{ctrl['port']}/onos/v1{endpoint}"
    t0 = time.time()
    try:
        r = requests.get(url, auth=AUTH, timeout=timeout)
        lat_ms = (time.time() - t0) * 1000
        rest_lat_hist[(ctrl['id'], endpoint)].append(lat_ms)
        return (r.json() if r.status_code == 200 else None, lat_ms)
    except Exception:
        rest_lat_hist[(ctrl['id'], endpoint)].append(-1)
        return (None, -1)

def safe_rate(current, previous, interval):
    if current is None or previous is None: return 0.0
    if pd.isna(current) or pd.isna(previous): return 0.0
    try:
        curr_val = float(current)
        prev_val = float(previous)
    except (ValueError, TypeError):
        return 0.0
    delta = curr_val - prev_val
    if delta < 0: return 0.0
    safe_interval = max(1, interval)
    rate = delta / safe_interval
    if rate > 1e9: return 0.0
    return rate

def reset_prev_state(reason=""):
    global prev_stats, prev_metrics, prev_flows_set
    prev_stats.clear()
    prev_metrics.clear()
    prev_flows_set.clear()
    rest_lat_hist.clear()
    attack_markers.append({
        "ts_utc": now_utc_iso(),
        "t_monotonic": time.monotonic() - monotonic_start,
        "event": "reset_prev_state",
        "reason": reason
    })
    print(f"[RESET] prev_* cleared. reason={reason}")

def get_docker_stats():
    try:
        p = subprocess.run(
            ['docker','stats','--no-stream','--format','{{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}'],
            capture_output=True, text=True, timeout=3
        )
        out = {}
        for line in p.stdout.strip().splitlines():
            parts = line.split('\t')
            if len(parts) >= 3:
                name = parts[0]
                try: cpu = float(parts[1].replace('%','').strip())
                except: cpu = 0.0
                mem = parts[2].split('/')[0].strip()
                out[name] = {"cpu": cpu, "mem": mem}
        return out
    except Exception:
        return {}

def get_hosts_mac_map():
    mac_map = {}
    for ctrl in CONTROLLERS:
        hosts_data, _ = query(ctrl, "/hosts")
        if hosts_data:
            for h in hosts_data.get("hosts", []):
                mac = (h.get("mac") or "").lower()
                ips = h.get("ipAddresses") or []
                if mac and ips:
                    mac_map[mac] = {"ip": ips[0]}
    return mac_map

def dump_switch_flows(sw):
    try:
        p = subprocess.run(['sudo','ovs-ofctl','-O','OpenFlow13','dump-flows',sw],
                           capture_output=True, text=True, timeout=5)
        return p.stdout
    except Exception:
        return ""

def generate_flow_id(switch, src_ip, dst_ip, src_port, dst_port, protocol):
    key = f"{switch}|{src_ip}|{dst_ip}|{src_port}|{dst_port}|{protocol}"
    return hashlib.md5(key.encode()).hexdigest()[:16]

def shannon_entropy(values):
    if not values: return 0.0
    total = sum(values)
    if total <= 0: return 0.0
    probs = [v/total for v in values if v > 0]
    return -sum(p*log2(p) for p in probs)

def parse_flows_enhanced(sw, text, mac_map):
    global flow_database
    flows = []
    if not text: return flows
    for line in text.strip().splitlines():
        if "n_packets=" not in line or "actions=" not in line: continue
        m_pkts = re.search(r'n_packets=(\d+)', line)
        if not m_pkts or int(m_pkts.group(1)) == 0: continue

        m_bytes = re.search(r'n_bytes=(\d+)', line)
        m_dur   = re.search(r'duration=(\d+(?:\.\d+)?)s', line)
        m_prio  = re.search(r'priority=(\d+)', line)
        m_cookie= re.search(r'cookie=(0x[0-9a-f]+)', line)

        m_srcmac= re.search(r'dl_src=([0-9a-f:]{17})', line, re.I)
        m_dstmac= re.search(r'dl_dst=([0-9a-f:]{17})', line, re.I)
        m_srcip = re.search(r'nw_src=([0-9.]+)', line)
        m_dstip = re.search(r'nw_dst=([0-9.]+)', line)

        m_proto = re.search(r'nw_proto=(\d+)', line)
        m_proto_token = re.search(r'\b(tcp|udp|icmp)\b', line, re.IGNORECASE)
        m_sport = re.search(r'tp_src=(\d+)', line)
        m_dport = re.search(r'tp_dst=(\d+)', line)

        m_tflags= re.search(r'tcp_flags=([^,\s]+)', line)
        m_itype = re.search(r'icmp_type=(\d+)', line)
        m_icode = re.search(r'icmp_code=(\d+)', line)
        m_inp   = re.search(r'in_port=(?:"([^"]+)|(\d+))', line)
        m_act   = re.search(r'actions=([^\s]+)', line)

        src_ip = m_srcip.group(1) if m_srcip else None
        dst_ip = m_dstip.group(1) if m_dstip else None
        if not src_ip and m_srcmac: src_ip = (mac_map.get(m_srcmac.group(1).lower()) or {}).get("ip")
        if not dst_ip and m_dstmac: dst_ip = (mac_map.get(m_dstmac.group(1).lower()) or {}).get("ip")
        if not (src_ip and dst_ip): continue

        proto_num = int(m_proto.group(1)) if m_proto else None
        if proto_num is not None:
            proto_map = {1: 'ICMP', 6: 'TCP', 17: 'UDP'}
            proto_name = proto_map.get(proto_num, f"other_{proto_num}")
        elif m_proto_token:
            token = m_proto_token.group(1).lower()
            token_map = {'tcp': ('TCP', 6), 'udp': ('UDP', 17), 'icmp': ('ICMP', 1)}
            proto_name, proto_num = token_map.get(token, (token.upper(), 0))
        else:
            proto_num = 0
            proto_name = 'OTHER'

        src_port  = int(m_sport.group(1)) if m_sport else 0
        dst_port  = int(m_dport.group(1)) if m_dport else 0

        fid = generate_flow_id(sw, src_ip, dst_ip, src_port, dst_port, proto_name)
        pkt = int(m_pkts.group(1))
        byt = int(m_bytes.group(1)) if m_bytes else 0
        dur = float(m_dur.group(1)) if m_dur else 0.0

        flow = {
            "flow_id": fid, "switch": sw,
            "cookie": m_cookie.group(1) if m_cookie else None,
            "priority": int(m_prio.group(1)) if m_prio else 0,
            "src_ip": src_ip, "dst_ip": dst_ip,
            "src_port": src_port, "dst_port": dst_port,
            "protocol": proto_name, "proto_num": proto_num,
            "packets": pkt, "bytes": byt, "duration": dur,
            "tcp_flags": m_tflags.group(1) if m_tflags else None,
            "icmp_type": int(m_itype.group(1)) if m_itype else None,
            "icmp_code": int(m_icode.group(1)) if m_icode else None,
            "in_port": (m_inp.group(1) if m_inp and m_inp.group(1) else
                        (int(m_inp.group(2)) if m_inp and m_inp.group(2) else None)),
            "actions": m_act.group(1) if m_act else None,
            "master_controller": SWITCH_TO_CONTROLLER.get(sw,"unknown"),
        }
        flows.append(flow)

        flow_database[fid] = {
            "switch": sw, "src_ip": src_ip, "dst_ip": dst_ip,
            "src_port": src_port, "dst_port": dst_port, "protocol": proto_name,
            "cookie": flow["cookie"], "priority": flow["priority"],
            "last_seen": now_utc_iso(), "total_packets": pkt, "total_bytes": byt
        }

    return flows

def get_onos_metrics_best_effort(ctrl):
    out = {"pktin_total": 0.0, "pktout_total": 0.0, "flowmiss_total": np.nan,
           "pktin_rate_direct": 0.0, "pktout_rate_direct": 0.0,
           "event_queue": np.nan, "atomix_nodes": np.nan, "gc_pause_ms": np.nan}

    mjson, _ = query(ctrl, "/metrics")
    if not mjson or not isinstance(mjson.get("metrics"), list):
        return out

    pktin_counters = []
    pktout_counters = []
    pktin_rates = []
    pktout_rates = []

    for m in mjson["metrics"]:
        name = (m.get("name") or "").lower()

        fval_count = None
        fval_mean_rate = None
        metric_obj = m.get('metric') if isinstance(m, dict) else None
        if isinstance(metric_obj, dict):
            meter = metric_obj.get('meter')
            if isinstance(meter, dict):
                fval_count = meter.get('counter')
                fval_mean_rate = meter.get('mean_rate')
            if fval_count is None and 'counter' in metric_obj and isinstance(metric_obj.get('counter'), dict):
                fval_count = metric_obj.get('counter').get('counter')

        if fval_count is None:
            if 'value' in m and m.get('value') is not None:
                try: fval_count = float(m.get('value'))
                except: fval_count = None
            elif 'measurements' in m and isinstance(m.get('measurements'), list) and m.get('measurements'):
                first = m.get('measurements')[0]
                if isinstance(first, dict):
                    fval_count = first.get('value') or first.get('count')
                else:
                    fval_count = first

        try:
            if fval_count is not None: fval_count = float(fval_count)
        except: fval_count = None
        try:
            if fval_mean_rate is not None: fval_mean_rate = float(fval_mean_rate)
        except: fval_mean_rate = None

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?in\.count', name):
            if fval_count is not None and fval_count > 0:
                pktin_counters.append(fval_count)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?in\.rate', name):
            if fval_mean_rate is not None and fval_mean_rate > 0:
                pktin_rates.append(fval_mean_rate)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?out\.count', name):
            if fval_count is not None and fval_count > 0:
                pktout_counters.append(fval_count)

        if re.search(r'of:[0-9a-f]+\.packet[_\.-]?out\.rate', name):
            if fval_mean_rate is not None and fval_mean_rate > 0:
                pktout_rates.append(fval_mean_rate)

        if re.search(r'flow[_\.-]?miss|miss[_\.-]?flow|flowmiss', name) and not re.search(r'rate', name):
            if fval_count is not None:
                out['flowmiss_total'] = fval_count

        if re.search(r'event[_\.-]?queue|queue[_\.-]?size', name):
            if fval_count is not None:
                out['event_queue'] = fval_count

        if 'atomix' in name and re.search(r'pending|backlog|queue', name):
            if fval_count is not None:
                out['atomix_nodes'] = fval_count

    if pktin_counters:
        out['pktin_total'] = sum(pktin_counters)
    if pktin_rates:
        out['pktin_rate_direct'] = sum(pktin_rates)
    if pktout_counters:
        out['pktout_total'] = sum(pktout_counters)
    if pktout_rates:
        out['pktout_rate_direct'] = sum(pktout_rates)

    fjson, _ = query(ctrl, "/flows")
    if fjson and isinstance(fjson.get("flows"), list):
        miss = sum(1 for f in fjson["flows"] if int(f.get("packets", 0)) == 0)
        if np.isnan(out.get("flowmiss_total", np.nan)):
            out["flowmiss_total"] = miss

    cjson, _ = query(ctrl, "/cluster")
    if cjson and isinstance(cjson.get("nodes"), list):
        out["atomix_nodes"] = sum(1 for n in cjson["nodes"] if n.get("status") == "READY")

    try:
        p = subprocess.run(['docker','logs','--tail','50', ctrl['id']],
                           capture_output=True, text=True, timeout=1.0)
        for line in (p.stderr or "").splitlines():
            if 'GC' in line and 'pause' in line.lower():
                m_gc = re.search(r'(\d+\.?\d*)\s*ms', line)
                if m_gc: out["gc_pause_ms"] = float(m_gc.group(1)); break
    except Exception:
        pass

    return out

def get_onos_flows_with_protocol(ctrl):
    proto_counts = {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0}
    fjson, _ = query(ctrl, "/flows")
    if not fjson or not isinstance(fjson.get("flows"), list):
        return proto_counts

    for flow in fjson["flows"]:
        packets = flow.get("packets", 0)
        if packets == 0:
            continue

        criteria = flow.get("selector", {}).get("criteria", [])
        proto_found = False
        is_tcp = False

        for c in criteria:
            if c.get("type") == "IP_PROTO":
                proto_num = c.get("protocol")
                if proto_num == 6:
                    proto_counts["tcp"] += 1
                    proto_found = True
                    is_tcp = True
                elif proto_num == 17:
                    proto_counts["udp"] += 1
                    proto_found = True
                elif proto_num == 1:
                    proto_counts["icmp"] += 1
                    proto_found = True
                break

        if is_tcp and packets <= 5:
            proto_counts["syn_flows"] += 1

        if not proto_found:
            for c in criteria:
                if c.get("type") == "ETH_TYPE" and c.get("ethType") in ["0x800", "0x0800"]:
                    proto_counts["other"] += 1
                    break

    return proto_counts


# ────────────────────────────────────────────────────────────────────
#  PER-SOURCE-IP FEATURES (Approach 1 — NEW in v1.4.0)
# ────────────────────────────────────────────────────────────────────
def compute_per_source_features(all_switch_flows):
    """
    Group flows by src_ip → compute per-source stats for top-K sources.
    Returns dict with fixed columns: src1_ip, src1_packets, ..., src6_syn_ratio, etc.
    Produces 10 features × TOP_K_SOURCES + 3 global = 63 new columns.
    """
    if not all_switch_flows:
        out = {}
        for i in range(1, TOP_K_SOURCES + 1):
            out[f'src{i}_ip'] = 'none'
            out[f'src{i}_packets'] = 0
            out[f'src{i}_bytes'] = 0
            out[f'src{i}_flows'] = 0
            out[f'src{i}_pps'] = 0.0
            out[f'src{i}_syn_ratio'] = 0.0
            out[f'src{i}_dst_diversity'] = 0
            out[f'src{i}_top_dst_share'] = 0.0
            out[f'src{i}_avg_duration'] = 0.0
            out[f'src{i}_pkt_share'] = 0.0
        out['active_sources'] = 0
        out['source_entropy'] = 0.0
        out['top_source_dominance'] = 0.0
        return out

    src_stats = defaultdict(lambda: {
        'packets': 0, 'bytes': 0, 'flows': 0, 'syn_flows': 0,
        'dsts': set(), 'dst_pkts': defaultdict(int), 'durations': [],
    })

    total_packets = 0
    for f in all_switch_flows:
        src = f.get('src_ip')
        if not src:
            continue
        s = src_stats[src]
        pkts = f.get('packets', 0)
        s['packets'] += pkts
        s['bytes'] += f.get('bytes', 0)
        s['flows'] += 1
        s['dsts'].add(f.get('dst_ip', ''))
        s['dst_pkts'][f.get('dst_ip', '')] += pkts
        dur = f.get('duration', 0.0)
        s['durations'].append(dur)
        total_packets += pkts
        # SYN heuristic: TCP flow with very few packets
        proto = f.get('protocol', '')
        if proto == 'TCP' and pkts <= 5:
            s['syn_flows'] += 1

    # Rank by packets descending
    ranked = sorted(src_stats.items(), key=lambda kv: kv[1]['packets'], reverse=True)

    out = {}
    for i in range(1, TOP_K_SOURCES + 1):
        if i <= len(ranked):
            src_ip, st = ranked[i - 1]
            top_dst_pkts = max(st['dst_pkts'].values()) if st['dst_pkts'] else 0
            out[f'src{i}_ip'] = src_ip
            out[f'src{i}_packets'] = st['packets']
            out[f'src{i}_bytes'] = st['bytes']
            out[f'src{i}_flows'] = st['flows']
            out[f'src{i}_pps'] = st['packets'] / max(1, INTERVAL)
            out[f'src{i}_syn_ratio'] = st['syn_flows'] / max(1, st['flows'])
            out[f'src{i}_dst_diversity'] = len(st['dsts'])
            out[f'src{i}_top_dst_share'] = top_dst_pkts / max(1, st['packets'])
            out[f'src{i}_avg_duration'] = float(np.mean(st['durations'])) if st['durations'] else 0.0
            out[f'src{i}_pkt_share'] = st['packets'] / max(1, total_packets)
        else:
            out[f'src{i}_ip'] = 'none'
            out[f'src{i}_packets'] = 0
            out[f'src{i}_bytes'] = 0
            out[f'src{i}_flows'] = 0
            out[f'src{i}_pps'] = 0.0
            out[f'src{i}_syn_ratio'] = 0.0
            out[f'src{i}_dst_diversity'] = 0
            out[f'src{i}_top_dst_share'] = 0.0
            out[f'src{i}_avg_duration'] = 0.0
            out[f'src{i}_pkt_share'] = 0.0

    # Global source diversity metrics
    out['active_sources'] = len(src_stats)
    pkt_counts = [st['packets'] for _, st in ranked]
    out['source_entropy'] = shannon_entropy(pkt_counts)
    out['top_source_dominance'] = (ranked[0][1]['packets'] / max(1, total_packets)) if ranked else 0.0

    return out


def compute_features(all_switch_flows, per_ctrl_stats, per_switch_stats, docker_stats, per_ctrl_metrics, per_ctrl_protos):
    global prev_stats, prev_flows_set, current_label, prev_metrics

    five_seen = set()
    dedup = []
    for f in all_switch_flows:
        key5 = (f["src_ip"], f["dst_ip"], f["src_port"], f["dst_port"], f["protocol"])
        if key5 in five_seen: continue
        five_seen.add(key5)
        dedup.append(f)
    flows = dedup

    utc_now = datetime.now(timezone.utc)
    features = {
        "schema": SCHEMA_VERSION,
        "timestamp_utc": utc_now.isoformat(),
        "timestamp_unix": utc_now.timestamp(),
        "t_monotonic": time.monotonic() - monotonic_start,
        "sample_id": len(collected_samples) + 1,
        "label": current_label,
        "attacker_ip": current_attacker_ip,
        "attacker_host": current_attacker_host
    }

    for ctrl in CONTROLLERS:
        cid = ctrl['id']
        agg = per_ctrl_stats.get(cid, {"flows":0,"packets":0,"bytes":0})
        features[f"{cid}_flows"]   = agg["flows"]
        features[f"{cid}_packets"] = agg["packets"]
        features[f"{cid}_bytes"]   = agg["bytes"]

        for k in ("flows","packets","bytes"):
            kk = f"{cid}_{k}"
            prev = prev_stats.get(kk)
            features[f"{kk}_rate"] = safe_rate(agg[k], prev, INTERVAL)
            prev_stats[kk] = agg[k]

        features[f"{cid}_cpu"] = (docker_stats.get(cid) or {}).get("cpu", 0.0)
        features[f"{cid}_mem"] = (docker_stats.get(cid) or {}).get("mem", "0B")

        m = per_ctrl_metrics.get(cid, {})
        for key in ("pktin_total","pktout_total","flowmiss_total"):
            v = m.get(key, np.nan)
            features[f"{cid}_{key}"] = v if not pd.isna(v) else 0.0
            prev_v = prev_metrics.get(f"{cid}_{key}")
            features[f"{cid}_{key}_rate"] = safe_rate(v, prev_v, INTERVAL)
            prev_metrics[f"{cid}_{key}"] = v if not pd.isna(v) else 0.0

        features[f"{cid}_event_queue"]  = m.get("event_queue", np.nan)
        features[f"{cid}_atomix_nodes"] = m.get("atomix_nodes", np.nan)
        features[f"{cid}_gc_pause_ms"]  = m.get("gc_pause_ms", np.nan)

        for ep in ("/flows","/hosts","/metrics"):
            hist = rest_lat_hist.get((cid, ep), [])
            if hist:
                clean = [x for x in hist if x >= 0]
                features[f"{cid}_rest{ep.replace('/','_')}_avg_ms"] = float(np.mean(clean)) if clean else np.nan
                features[f"{cid}_rest{ep.replace('/','_')}_max_ms"] = float(np.max(clean)) if clean else np.nan
            else:
                features[f"{cid}_rest{ep.replace('/','_')}_avg_ms"] = np.nan
                features[f"{cid}_rest{ep.replace('/','_')}_max_ms"] = np.nan

        proto = per_ctrl_protos.get(cid, {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0})
        features[f"{cid}_tcp_flows"] = proto["tcp"]
        features[f"{cid}_udp_flows"] = proto["udp"]
        features[f"{cid}_icmp_flows"] = proto["icmp"]
        features[f"{cid}_other_flows"] = proto["other"]
        features[f"{cid}_syn_flows"] = proto["syn_flows"]

    for sw in SWITCHES:
        s = per_switch_stats.get(sw, {"flows":0,"packets":0,"bytes":0})
        features[f"{sw}_flows"] = s["flows"]
        features[f"{sw}_packets"] = s["packets"]
        features[f"{sw}_bytes"] = s["bytes"]
        features[f"{sw}_master"] = SWITCH_TO_CONTROLLER.get(sw,"unknown")

    if flows:
        df = pd.DataFrame(flows)
        features["total_flows"]   = len(df)
        features["total_packets"] = int(df["packets"].sum())
        features["total_bytes"]   = int(df["bytes"].sum())
        features["avg_pkt_size"]  = features["total_bytes"]/max(1,features["total_packets"])

        counts = df["protocol"].value_counts()
        features["tcp_flows"] = int(counts.get("TCP",0))
        features["udp_flows"] = int(counts.get("UDP",0))
        features["icmp_flows"]= int(counts.get("ICMP",0))
        total = max(1,features["total_flows"])
        features["tcp_ratio"]  = features["tcp_flows"]/total
        features["udp_ratio"]  = features["udp_flows"]/total
        features["icmp_ratio"] = features["icmp_flows"]/total

        cluster_tcp = sum(per_ctrl_protos.get(c["id"], {}).get("tcp", 0) for c in CONTROLLERS)
        cluster_udp = sum(per_ctrl_protos.get(c["id"], {}).get("udp", 0) for c in CONTROLLERS)
        cluster_icmp = sum(per_ctrl_protos.get(c["id"], {}).get("icmp", 0) for c in CONTROLLERS)
        cluster_other = sum(per_ctrl_protos.get(c["id"], {}).get("other", 0) for c in CONTROLLERS)
        cluster_syn = sum(per_ctrl_protos.get(c["id"], {}).get("syn_flows", 0) for c in CONTROLLERS)
        cluster_total = max(1, cluster_tcp + cluster_udp + cluster_icmp + cluster_other)
        features["cluster_tcp_flows"] = cluster_tcp
        features["cluster_udp_flows"] = cluster_udp
        features["cluster_icmp_flows"] = cluster_icmp
        features["cluster_other_flows"] = cluster_other
        features["cluster_syn_flows"] = cluster_syn
        features["cluster_tcp_ratio"] = cluster_tcp / cluster_total
        features["cluster_udp_ratio"] = cluster_udp / cluster_total
        features["cluster_icmp_ratio"] = cluster_icmp / cluster_total
        features["cluster_syn_ratio"] = cluster_syn / max(1, cluster_tcp) if cluster_tcp > 0 else 0.0

        features["unique_srcs"] = df["src_ip"].nunique()
        features["unique_dsts"] = df["dst_ip"].nunique()
        features["src_dst_ratio"] = features["unique_srcs"]/max(1,features["unique_dsts"])
        features["src_entropy"] = shannon_entropy(df.groupby("src_ip")["packets"].sum().tolist())
        features["dst_entropy"] = shannon_entropy(df.groupby("dst_ip")["packets"].sum().tolist())

        features["unique_dst_ports"] = df["dst_port"].nunique()
        features["unique_src_ports"] = df["src_port"].nunique()
        features["port_diversity"] = (features["unique_src_ports"] + features["unique_dst_ports"])/2

        top_dst = df.groupby("dst_ip")["packets"].sum().idxmax()
        top_dst_pkts = int(df[df["dst_ip"] == top_dst]["packets"].sum())
        features["top_dst"] = top_dst
        features["top_dst_packets"] = top_dst_pkts
        features["top_dst_share"] = top_dst_pkts/max(1,features["total_packets"])

        curr = set(df["flow_id"])
        new_flows = len(curr - prev_flows_set)
        features["flow_arrival_rate"] = new_flows/max(1,INTERVAL)
        prev_flows_set.clear(); prev_flows_set.update(curr)

        features["avg_flow_duration"] = float(df["duration"].mean())
        features["max_flow_duration"] = float(df["duration"].max())
        features["min_flow_duration"] = float(df["duration"].min())
        features["avg_pkts_per_flow"] = features["total_packets"]/max(1,features["total_flows"])
        features["max_pkts_per_flow"] = int(df["packets"].max())
        stdv = df["packets"].std()
        features["std_pkts_per_flow"] = float(stdv if not pd.isna(stdv) else 0.0)

    else:
        features.update({
            "total_flows":0,"total_packets":0,"total_bytes":0,"avg_pkt_size":0.0,
            "tcp_flows":0,"udp_flows":0,"icmp_flows":0,"tcp_ratio":0.0,"udp_ratio":0.0,"icmp_ratio":0.0,
            "cluster_tcp_flows":0,"cluster_udp_flows":0,"cluster_icmp_flows":0,"cluster_other_flows":0,
            "cluster_syn_flows":0,"cluster_tcp_ratio":0.0,"cluster_udp_ratio":0.0,"cluster_icmp_ratio":0.0,"cluster_syn_ratio":0.0,
            "unique_srcs":0,"unique_dsts":0,"src_dst_ratio":0.0,"src_entropy":0.0,"dst_entropy":0.0,
            "unique_dst_ports":0,"unique_src_ports":0,"port_diversity":0.0,
            "top_dst":"none","top_dst_packets":0,"top_dst_share":0.0,
            "flow_arrival_rate":0.0,
            "avg_flow_duration":0.0,"max_flow_duration":0.0,"min_flow_duration":0.0,
            "avg_pkts_per_flow":0.0,"max_pkts_per_flow":0,"std_pkts_per_flow":0.0
        })

    ctrl_loads = [per_ctrl_stats.get(c["id"],{}).get("packets",0) for c in CONTROLLERS]
    sw_loads   = [per_switch_stats.get(sw,{}).get("packets",0) for sw in SWITCHES]
    features["ctrl_load_std"]  = float(np.std(ctrl_loads)) if ctrl_loads else 0.0
    features["ctrl_load_mean"] = float(np.mean(ctrl_loads)) if ctrl_loads else 0.0
    features["ctrl_load_max"]  = int(max(ctrl_loads)) if ctrl_loads else 0
    features["ctrl_load_min"]  = int(min(ctrl_loads)) if ctrl_loads else 0
    features["switch_load_std"]= float(np.std(sw_loads)) if sw_loads else 0.0
    features["switch_load_mean"]=float(np.mean(sw_loads)) if sw_loads else 0.0

    features["cluster_size"] = len(CONTROLLERS)
    features["active_controllers"] = sum(1 for c in CONTROLLERS if per_ctrl_stats.get(c["id"]))

    features["tracked_flows"] = len(flow_database)
    features["attack_in_progress"] = 1 if current_label==1 else 0
    features["cfg_interval_s"] = INTERVAL
    features["cfg_switches"] = ",".join(SWITCHES)
    features["cfg_controllers"] = ",".join(c["id"] for c in CONTROLLERS)

    features["warmup"] = 1 if features["sample_id"] <= WARMUP_SAMPLES else 0

    # ── Per-source-IP features (Approach 1 — NEW) ──
    features.update(compute_per_source_features(all_switch_flows))

    return features

def monitoring_loop():
    global monitoring_active, collected_samples, current_label, parquet_pending_counter, current_attacker_ip, current_attacker_host
    last_label_mtime = 0
    label_file = '/tmp/label_state'

    while monitoring_active:
        t_start_loop = time.time()
        old_label = current_label

        try:
            try:
                if os.path.exists(label_file):
                    mtime = os.path.getmtime(label_file)
                    if mtime > last_label_mtime:
                        with open(label_file, 'r') as f:
                            lines = f.readlines()
                            if lines:
                                last_line = lines[-1].strip()
                                parts = last_line.split()
                                if len(parts) >= 1:
                                    try:
                                        new_label = int(parts[0])
                                        if new_label != current_label:
                                            current_label = new_label
                                            print(f"[Label Sync] Changed to {current_label}")
                                        # Parse attacker identity from label_state
                                        if new_label == 1 and len(parts) >= 3:
                                            current_attacker_ip = parts[2].strip()
                                            current_attacker_host = parts[1].split("_")[0]
                                            print(f"[Label Sync] Attacker: {current_attacker_host} ({current_attacker_ip})")
                                        elif new_label == 1 and len(parts) >= 2:
                                            host_tag = parts[1].split("_")[0]
                                            current_attacker_ip = HOST_TO_IP.get(host_tag, "none")
                                            current_attacker_host = host_tag
                                            print(f"[Label Sync] Attacker inferred: {host_tag} ({current_attacker_ip})")
                                        elif new_label == 0:
                                            current_attacker_ip = "none"
                                            current_attacker_host = "none"
                                    except ValueError:
                                        pass
                        last_label_mtime = mtime
            except Exception as e:
                print(f"[Label Sync Error] {e}")

            if current_label != old_label:
                reset_prev_state(reason=f"label_flip {old_label}->{current_label}")

            mac_map = get_hosts_mac_map()
            docker = get_docker_stats()

            per_switch_stats = {}
            per_ctrl_stats = {}
            all_switch_flows = []

            for sw in SWITCHES:
                text = dump_switch_flows(sw)
                flows = parse_flows_enhanced(sw, text, mac_map)
                all_switch_flows.extend(flows)

                s_flows = len(flows)
                s_packets = sum(f.get("packets", 0) for f in flows)
                s_bytes = sum(f.get("bytes", 0) for f in flows)
                per_switch_stats[sw] = {"flows": s_flows, "packets": s_packets, "bytes": s_bytes}

                ctrl_ip = SWITCH_TO_CONTROLLER.get(sw)
                ctrl_id = next((c["id"] for c in CONTROLLERS if c["ip"] == ctrl_ip), None)
                if ctrl_id:
                    agg = per_ctrl_stats.setdefault(ctrl_id, {"flows": 0, "packets": 0, "bytes": 0})
                    agg["flows"] += s_flows
                    agg["packets"] += s_packets
                    agg["bytes"] += s_bytes

            per_ctrl_metrics = {}
            per_ctrl_protos = {}
            for ctrl in CONTROLLERS:
                try:
                    per_ctrl_metrics[ctrl["id"]] = get_onos_metrics_best_effort(ctrl)
                    per_ctrl_protos[ctrl["id"]] = get_onos_flows_with_protocol(ctrl)
                except Exception as e:
                    print(f"[monitor] Error querying {ctrl['id']}: {e}")
                    per_ctrl_metrics[ctrl["id"]] = {"pktin_total": 0.0, "pktout_total": 0.0}
                    per_ctrl_protos[ctrl["id"]] = {"tcp": 0, "udp": 0, "icmp": 0, "other": 0, "syn_flows": 0}

            features = compute_features(all_switch_flows, per_ctrl_stats, per_switch_stats, docker, per_ctrl_metrics, per_ctrl_protos)

            with data_lock:
                collected_samples.append(features)
                parquet_pending_counter += 1
                if len(collected_samples) > 10000:
                    collected_samples.pop(0)

            training_queue.put(features)

            try:
                df = pd.DataFrame([features])
                header = not os.path.exists(CSV_PATH)
                df.to_csv(CSV_PATH, mode='a', header=header, index=False)
            except Exception as e:
                print("[monitor] CSV save failed:", e)

            if parquet_pending_counter >= PARQUET_SYNC_EVERY:
                try:
                    import pyarrow
                    with data_lock:
                        dfp = pd.DataFrame(collected_samples)
                        parquet_pending_counter = 0
                    dfp.to_parquet(PARQUET_PATH, index=False)
                except Exception:
                    pass

        except Exception as e:
            print("[monitor] loop error:", e)

        elapsed = time.time() - t_start_loop
        sleep_time = max(0, INTERVAL - elapsed)
        if monitoring_active and sleep_time > 0:
            time.sleep(sleep_time)

def start_monitoring(label=0):
    global monitoring_active, monitoring_thread, current_label
    if monitoring_active:
        print("Monitoring already running")
        return
    current_label = int(label)
    monitoring_active = True
    reset_prev_state(reason="start_monitoring")
    monitoring_thread = threading.Thread(target=monitoring_loop, daemon=True)
    monitoring_thread.start()
    print(f"✓ Monitoring started (label={current_label}). Sampling every {INTERVAL}s")

def stop_monitoring():
    global monitoring_active, monitoring_thread, parquet_pending_counter
    if not monitoring_active:
        print("Monitoring is not running")
        return
    monitoring_active = False
    if monitoring_thread is not None:
        monitoring_thread.join(timeout=10)
    with data_lock:
        df_all = pd.DataFrame(collected_samples)

    # NEW (appends):
    try:
        header = not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0
        df_all.to_csv(CSV_PATH, mode='a', header=header, index=False)
    except Exception:
        pass

    try:
        import pyarrow
        df_all.to_parquet(PARQUET_PATH, index=False)
        parquet_pending_counter = 0
    except Exception:
        pass
    print(f"✓ Monitoring stopped. {len(collected_samples)} samples saved to: {CSV_PATH}")

def set_label(label):
    global current_label
    old = current_label
    current_label = int(label)
    print(f"✓ Label set to {current_label}")
    if current_label != old:
        reset_prev_state(reason=f"manual_label {old}->{current_label}")

def show_status(verbose=True, top_n=10):
    print("=" * 60)
    print(f"MONITORING STATUS (schema v{SCHEMA_VERSION})")
    print("=" * 60)
    print(f"Active: {monitoring_active}")
    with data_lock:
        print(f"Collected samples (in memory): {len(collected_samples)}")
        last = collected_samples[-1] if collected_samples else None
    print(f"Training queue size: {training_queue.qsize()}")

    if not last:
        print("No samples recorded yet.")
        return

    print("\n" + "-" * 60)
    print("LAST SAMPLE")
    print("-" * 60)
    print(f"Timestamp: {last.get('timestamp_utc')}")
    print(f"Sample ID: {last.get('sample_id')} | Label: {last.get('label')} | Warmup: {last.get('warmup')}")
    print(f"Total flows: {last.get('total_flows', 0)} | Total packets: {last.get('total_packets', 0)}")
    # Show per-source summary
    print(f"Active sources: {last.get('active_sources', 0)} | Source entropy: {last.get('source_entropy', 0):.2f}")
    for i in range(1, min(4, TOP_K_SOURCES + 1)):
        ip = last.get(f'src{i}_ip', 'none')
        pkts = last.get(f'src{i}_packets', 0)
        share = last.get(f'src{i}_pkt_share', 0)
        syn = last.get(f'src{i}_syn_ratio', 0)
        dst_div = last.get(f'src{i}_dst_diversity', 0)
        if ip != 'none':
            print(f"  src{i}: {ip} | pkts={pkts} share={share:.2f} syn_ratio={syn:.2f} dst_div={dst_div}")
    print("=" * 60)

def list_active_flows(limit=20):
    items = sorted(flow_database.items(), key=lambda kv: kv[1].get("total_packets", 0), reverse=True)
    for fid, info in items[:limit]:
        print(fid, info.get("switch"), info.get("src_ip"), "->", info.get("dst_ip"), "pkts=", info.get("total_packets",0))

def get_flow_info(flow_id):
    return flow_database.get(flow_id)

print("=" * 70)
print("✓ SDN Controller Monitoring API v1.4.0 - READY (Per-Source-IP Enriched)")
print("=" * 70)
print("Functions: start_monitoring(label=0), stop_monitoring(), set_label(label)")
print("           show_status(verbose=True), list_active_flows(limit=20)")
print("           get_flow_info(flow_id)")
print("=" * 70)
print(f"REST_LAT_WINDOW={REST_LAT_WINDOW} | WARMUP_SAMPLES={WARMUP_SAMPLES}")
print(f"TOP_K_SOURCES={TOP_K_SOURCES} (63 new per-source columns)")
print("NEW: Per-source-IP features for DRL attacker identification (Approach 1)")
print("     Existing 162 columns UNCHANGED — LSTM works without modification")
print("=" * 70)


✓ SDN Controller Monitoring API v1.4.0 - READY (Per-Source-IP Enriched)
Functions: start_monitoring(label=0), stop_monitoring(), set_label(label)
           show_status(verbose=True), list_active_flows(limit=20)
           get_flow_info(flow_id)
REST_LAT_WINDOW=2 | WARMUP_SAMPLES=0
TOP_K_SOURCES=6 (63 new per-source columns)
NEW: Per-source-IP features for DRL attacker identification (Approach 1)
     Existing 162 columns UNCHANGED — LSTM works without modification


In [41]:
start_monitoring(label=0)

[RESET] prev_* cleared. reason=start_monitoring
✓ Monitoring started (label=0). Sampling every 2s
[Label Sync] Changed to 1
[Label Sync] Attacker: iot2 (10.10.0.102)
[RESET] prev_* cleared. reason=label_flip 0->1


[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0
[Label Sync] Changed to 1
[Label Sync] Attacker: iot2 (10.10.0.102)
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0
[Label Sync] Changed to 1
[Label Sync] Attacker: h2 (10.10.0.2)
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0
[Label Sync] Changed to 1
[Label Sync] Attacker: h4 (10.10.0.4)
[RESET] prev_* cleared. reason=label_flip 0->1
[Label Sync] Changed to 0
[RESET] prev_* cleared. reason=label_flip 1->0


In [42]:
# =====================================================================
# LIVE DRL NO-MITIGATION ENGINE (LSTM + DQN)
# =====================================================================
import os
import time
import json
import pickle
from pathlib import Path
from collections import deque
from datetime import datetime

import numpy as np
import pandas as pd
import psutil
import requests
import threading
import tensorflow as tf
from stable_baselines3 import DQN
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, average_precision_score
)

# =====================================================================
# 🛠️ RUNTIME CONTROLS
# =====================================================================
ACTIVE_MITIGATION = True    # 🔒 ENABLED — pushes drop rules to ONOS
DRL_DURATION      = 600     # Run duration in seconds
INTERVAL          = 2       # Loop polling interval

# ---------------------------- Resolve paths ----------------------------
CANDIDATE_BASES = [
    Path.cwd(),
    Path.cwd() / "Models/newTrail",
    Path("/home/shokran/PycharmProjects/JupyterProject/Models/newTrail"),
]
BASE = None
for b in CANDIDATE_BASES:
    if (b / "outputs_detector" / "config.json").exists():
        BASE = b
        break
if BASE is None:
    raise RuntimeError("Could not find outputs_detector/config.json. Set BASE manually.")

DETECTOR_DIR = BASE / "outputs_detector"
DRL_DIR      = BASE / "outputs_ppo_dqn_compare"

# ── Output folder: one for each mitigation mode ──────────────────────
_RUN_MODE = "active_mitigation" if ACTIVE_MITIGATION else "no_mitigation"
SAVE_DIR  = BASE / f"outputs_drl_{_RUN_MODE}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"[ENGINE] Output folder: {SAVE_DIR}")

if "data_lock" not in globals() or "collected_samples" not in globals():
    raise RuntimeError("Run your monitoring/collector cell first!")

# =====================================================================
# 🌐 ONOS / OVS CONFIGURATION
# =====================================================================
SWITCH_DPIDS = [f"of:{i:016x}" for i in range(1, 5)]
ONOS_NODES = [
    {'id': 'onos1', 'ip': '175.24.1.5', 'port': 8181},
    {'id': 'onos2', 'ip': '175.24.1.6', 'port': 8181},
    {'id': 'onos3', 'ip': '175.24.1.7', 'port': 8181},
]
ONOS_AUTH = ('onos', 'rocks')
CONTROLLER_IPS = {'175.24.1.5', '175.24.1.6', '175.24.1.7'}

def get_master_controller(device_id):
    for ctrl in ONOS_NODES:
        try:
            url = f"http://{ctrl['ip']}:{ctrl['port']}/onos/v1/devices/{device_id}"
            r = requests.get(url, auth=ONOS_AUTH, timeout=1)
            if r.ok and (r.json().get('role') or '').upper() == 'MASTER':
                return ctrl
        except Exception:
            continue
    return ONOS_NODES[0]

# ── Sentinel values that are never real IPs — always skip these ───────
_IP_SENTINELS = {"none", "None", "", "0.0.0.0", "nan"}

_ip_action_log = []   # list of dicts appended in the main loop

# =====================================================================
# 🧠 LOAD ARCHITECTURES
# =====================================================================
with open(DETECTOR_DIR / "config.json", "r") as f:
    det_cfg = json.load(f)

LSTM_FEATURES  = det_cfg["feature_cols"]
N_LSTM_FEATS   = int(det_cfg.get("n_features", len(LSTM_FEATURES)))
WINDOW_LEN     = int(det_cfg["BEST_WINDOW_LEN"])
BEST_THRESHOLD = float(det_cfg["BEST_THRESHOLD"])

with open(DETECTOR_DIR / "calibration.json", "r") as f:
    TEMPERATURE = float(json.load(f)["temperature"])
with open(DETECTOR_DIR / "imputer.pkl", "rb") as f:
    lstm_imputer = pickle.load(f)
with open(DETECTOR_DIR / "scaler.pkl", "rb") as f:
    lstm_scaler = pickle.load(f)

tf.get_logger().setLevel("ERROR")
lstm_model = tf.keras.models.load_model(DETECTOR_DIR / "model_final.keras", compile=False)

with open(DRL_DIR / "deployment_recipe.json", "r") as f:
    recipe = json.load(f)
with open(DRL_DIR / "obs_scaler.pkl", "rb") as f:
    dqn_scaler = pickle.load(f)

dqn_agent = DQN.load(str(DRL_DIR / "dqn_model.zip"))
TOP_K = int(recipe.get("top_k", 6))

SRC_NUM_FEATS  = ["packets", "bytes", "flows", "pps", "syn_ratio",
                  "dst_diversity", "top_dst_share", "avg_duration", "pkt_share"]
GLOBAL_FEATS   = ["active_sources", "source_entropy", "top_source_dominance"]

GT_MODE = "sample_or_file"

RUN_START_TIME = datetime.now().isoformat()

print(f"[ENGINE] Stage 1 (LSTM) Ready. Window={WINDOW_LEN}, Thresh={BEST_THRESHOLD:.4f}, T={TEMPERATURE:.4f}")
print(f"[ENGINE] Stage 2 (DQN) Action Space: 0=ALLOW, 1..{TOP_K}=BLOCK_SRC")
print(f"[ENGINE] Switches: {SWITCH_DPIDS}")
print(f"[ENGINE] ⚠️  ACTIVE_MITIGATION={ACTIVE_MITIGATION}")


# =====================================================================
# 🛡️ MITIGATION STATE TRACKER (with ONOS active mitigation logic)
# =====================================================================
class MitigationState:
    def __init__(self):
        self.active_action  = 0
        self.active_ip      = "none"
        self.applied_at     = None
        self.history        = []
        self.lock           = threading.RLock()

    def _install_drop(self, ip):
        if not ip or ip in _IP_SENTINELS or ip in CONTROLLER_IPS: return
        success_count = 0
        with self.lock:
            for device_id in SWITCH_DPIDS:
                master = get_master_controller(device_id)
                flow = {
                    'priority': 65000, 'isPermanent': True, 'deviceId': device_id,
                    'treatment': {'instructions': []},
                    'selector': {'criteria': [
                        {'type': 'ETH_TYPE', 'ethType': '0x0800'},
                        {'type': 'IPV4_SRC', 'ip': f'{ip}/32'}
                    ]}
                }
                try:
                    url = f"http://{master['ip']}:{master['port']}/onos/v1/flows"
                    if requests.post(url, json={'flows': [flow]}, auth=ONOS_AUTH, timeout=1).ok:
                        success_count += 1
                except: pass
        if success_count > 0:
            print(f"[MITIGATION] 🛡️  [DROP] Installed on {success_count}/{len(SWITCH_DPIDS)} switches for {ip}")

    def _clear_drop_rules(self, ip=None):
        target_ip = ip or self.active_ip
        if not target_ip or target_ip in _IP_SENTINELS: return
        removed = 0
        with self.lock:
            for device_id in SWITCH_DPIDS:
                master = get_master_controller(device_id)
                try:
                    url = f"http://{master['ip']}:{master['port']}/onos/v1/flows/{device_id}"
                    r = requests.get(url, auth=ONOS_AUTH, timeout=1)
                    if not r.ok: continue
                    for flow in r.json().get('flows', []):
                        if int(flow.get('priority', 0)) != 65000: continue
                        criteria = flow.get('selector', {}).get('criteria', []) or []
                        if any(c.get('type') == 'IPV4_SRC' and c.get('ip', '').split('/')[0] == target_ip for c in criteria):
                            fid = flow.get('id')
                            if fid and requests.delete(f"{url}/{fid}", auth=ONOS_AUTH, timeout=1).ok:
                                removed += 1
                except: pass
        if removed > 0:
            print(f"[MITIGATION] 🧹 Removed {removed} DROP flow(s) for {target_ip}")

    def _apply_rate_limit(self, ip):
        pass # Optional for later if needed

    def _remove_rate_limit(self):
        pass # Optional for later if needed

    def clear_all(self, step, reason="label_back_to_0"):
        if self.active_action == 0:
            return
        old = self.active_action
        self.history.append((step, old, 0, self.active_ip, reason))
        
        if ACTIVE_MITIGATION:
            if old == 2: self._clear_drop_rules(self.active_ip)
            print(f"[MITIGATION] 🧹 CLEARED action={old} on {self.active_ip} — reason: {reason}")
        else:
            print(f"[MITIGATION] 🧹 WOULD_CLEAR action={old} on {self.active_ip} — reason: {reason}")
            
        self.active_action = 0
        self.active_ip     = "none"
        self.applied_at    = None

    def update(self, new_action, target_ip, step):
        old = self.active_action

        if old == 2:
            if new_action != old:
                self.history.append((step, old, old, target_ip, f"blocked_downgrade_from_BLOCK_agent_wanted_{new_action}"))
            return

        if old == 1:
            if new_action == 2:
                if ACTIVE_MITIGATION:
                    print(f"[MITIGATION] 🔥 ESCALATING to BLOCK on {target_ip} (was RATE_LIMIT)")
                    self._install_drop(target_ip)
                else:
                    print(f"[MITIGATION] 🔥 WOULD_ESCALATE to BLOCK on {target_ip} (was RATE_LIMIT)")
                self.history.append((step, old, 2, target_ip, "escalation_rl_to_block"))
                self.active_action = 2
                self.active_ip     = target_ip
                self.applied_at    = time.time()
            else:
                if new_action != old:
                    self.history.append((step, old, old, target_ip, f"blocked_downgrade_from_RL_agent_wanted_{new_action}"))
            return

        if new_action == 1:
            print(f"[MITIGATION] {'🔄 RATE_LIMITING' if ACTIVE_MITIGATION else '🔄 WOULD_RATE_LIMIT'} {target_ip}")
            self.history.append((step, old, 1, target_ip, "agent_decision"))
            self.active_action = 1
            self.active_ip     = target_ip
            self.applied_at    = time.time()
        elif new_action == 2:
            if ACTIVE_MITIGATION:
                print(f"[MITIGATION] 🔥 BLOCKING {target_ip}")
                self._install_drop(target_ip)
            else:
                print(f"[MITIGATION] 🔥 WOULD_BLOCK {target_ip}")
            self.history.append((step, old, 2, target_ip, "agent_decision"))
            self.active_action = 2
            self.active_ip     = target_ip
            self.applied_at    = time.time()


# =====================================================================
# ⚙️ HELPERS & PIPELINE
# =====================================================================
def apply_feature_mapping(df):
    return df

def preprocess_live_sample(raw_df):
    df = raw_df.copy()
    df = apply_feature_mapping(df)
    for c in LSTM_FEATURES:
        if c not in df.columns:
            df[c] = 0.0
    for c in LSTM_FEATURES:
        if df[c].dtype == "object":
            ex = df[c].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0]
            df[c] = pd.to_numeric(ex, errors="coerce")
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    X = df[LSTM_FEATURES].values.astype(np.float64)
    if lstm_imputer is not None: X = lstm_imputer.transform(X)
    if lstm_scaler is not None:  X = lstm_scaler.transform(X)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def calibrate_probability(raw_prob, temperature):
    eps = 1e-7
    logits = np.log(np.clip(raw_prob, eps, 1 - eps) / np.clip(1 - raw_prob, eps, 1 - eps))
    return float(1.0 / (1.0 + np.exp(-logits / temperature)))

def read_label_state():
    try:
        with open("/tmp/label_state", "r") as f:
            lines = f.readlines()
            if lines:
                parts = lines[-1].strip().split()
                if int(parts[0]) == 1:
                    return 1, (parts[1].split("_")[0] if len(parts) >= 2 else "unknown"), \
                               (parts[2].strip() if len(parts) >= 3 else "none")
    except:
        pass
    return 0, "none", "none"

def resolve_ground_truth(sample):
    file_label, file_host, file_ip = read_label_state()
    try:
        sample_label = int(sample.get("label"))
    except Exception:
        sample_label = None

    if GT_MODE == "file_only" or sample_label not in (0, 1):
        return file_label, file_host, file_ip, file_label, sample_label, "file"

    if sample_label == 0:
        return 0, "none", "none", file_label, sample_label, "sample"

    sample_host = str(sample.get("label_host", sample.get("host", file_host)) or file_host)
    sample_ip   = str(sample.get("label_ip", file_ip) or file_ip)
    return 1, sample_host, sample_ip, file_label, sample_label, "sample"

def build_dqn_obs(attack_prob, sample):
    src_raw = np.array(
        [float(sample.get(f"src{k}_{f}", 0) or 0)
         for k in range(1, TOP_K + 1) for f in SRC_NUM_FEATS],
        dtype=np.float32
    )
    glob_raw = np.array([float(sample.get(g, 0) or 0) for g in GLOBAL_FEATS], dtype=np.float32)
    extra = np.concatenate([src_raw, glob_raw]).reshape(1, -1)
    extra = np.nan_to_num(extra, nan=0.0, posinf=1e6, neginf=-1e6)
    extra_scaled = np.clip(dqn_scaler.transform(extra), -10, 10).flatten()

    severity   = min(1.0, attack_prob * 1.2)
    confidence = abs(attack_prob - 0.5) * 2
    return np.concatenate([[attack_prob, severity, confidence], extra_scaled]).astype(np.float32)

def safe_get_docker_stats():
    fn = globals().get("get_docker_stats", None)
    return fn() if callable(fn) else {}

# =====================================================================
# 🚀 MAIN LOOP
# =====================================================================
feature_window = deque(maxlen=WINDOW_LEN)
step_log = []
prev_samples = step = 0
t_start = time.time()
process = psutil.Process(os.getpid())
_ = process.cpu_percent(interval=None)

mitigation = MitigationState()

# --- Running metrics ---
attack_episode_id        = 0
in_attack                = False
episode_start_step       = None
prob_at_mitigation_start = None
prev_gt_label            = 0
episode_probs            = []
policy_reactions         = []
detection_latencies      = []
attack_start_elapsed     = None
first_detection_elapsed  = None
first_policy_block_elapsed = None

docker_cache    = {}
docker_cache_ts = 0.0
DOCKER_CACHE_SEC = 2.0

print("\n====================================================")
print(f"🚀 LIVE DQN MODE (Active Mitigation: {ACTIVE_MITIGATION})")
print("====================================================\n")

try:
    while time.time() - t_start < DRL_DURATION:
        time.sleep(INTERVAL)

        with data_lock:
            snap = list(collected_samples)
        if len(snap) - prev_samples <= 0:
            continue

        new_samples = snap[prev_samples:]
        prev_samples = len(snap)
        elapsed = round(time.time() - t_start, 1)

        step_t0 = time.perf_counter()

        t_pre   = time.perf_counter()
        X_new   = preprocess_live_sample(pd.DataFrame(new_samples))
        preprocess_ms = (time.perf_counter() - t_pre) * 1000.0

        for row in X_new:
            feature_window.append(row)

        if len(feature_window) < WINDOW_LEN:
            print(f"[Buffer Accumulating] {len(feature_window)}/{WINDOW_LEN}")
            continue

        sample    = new_samples[-1]
        sample_id = int(sample.get("sample_id", step)) \
                    if str(sample.get("sample_id", "")).isdigit() else step

        gt_label, gt_host, gt_ip, gt_file_label, gt_sample_label, gt_source = \
            resolve_ground_truth(sample)
        gt_mismatch = int((gt_sample_label in (0, 1)) and (gt_sample_label != gt_file_label))

        if gt_label == 1 and prev_gt_label == 0:
            in_attack = True
            attack_episode_id += 1
            episode_start_step       = step
            episode_probs            = []
            prob_at_mitigation_start = None
            attack_start_elapsed     = elapsed
            first_detection_elapsed  = None
            first_policy_block_elapsed = None
            print(f"\n{'='*60}")
            print(f"⚠️  ATTACK EPISODE #{attack_episode_id} STARTED at step {step}")
            print(f"{'='*60}")

        elif gt_label == 0 and prev_gt_label == 1:
            print(f"\n{'='*60}")
            print(f"✅ ATTACK EPISODE #{attack_episode_id} ENDED at step {step}")
            if first_policy_block_elapsed is not None and attack_start_elapsed is not None:
                reaction = first_policy_block_elapsed - attack_start_elapsed
                policy_reactions.append(reaction)
                print(f"   Policy reaction time: {reaction:.1f}s")
            if first_detection_elapsed is not None and attack_start_elapsed is not None:
                det_lat = first_detection_elapsed - attack_start_elapsed
                detection_latencies.append(det_lat)
                print(f"   Detection latency:    {det_lat:.1f}s")
            print(f"{'='*60}")
            mitigation.clear_all(step, reason="attack_ended_gt_0")
            in_attack              = False
            attack_start_elapsed   = None
            first_detection_elapsed = None
            first_policy_block_elapsed = None

        prev_gt_label = gt_label

        # Stage 1: LSTM Perception
        window_arr = np.array(feature_window, dtype=np.float32).reshape(1, WINDOW_LEN, N_LSTM_FEATS)
        t_lstm = time.perf_counter()
        raw_prob = float(lstm_model.predict(window_arr, verbose=0).ravel()[0])
        cal_prob = calibrate_probability(raw_prob, TEMPERATURE)
        binary   = int(cal_prob >= BEST_THRESHOLD)
        lstm_ms  = (time.perf_counter() - t_lstm) * 1000.0

        if gt_label == 1 and cal_prob >= BEST_THRESHOLD and first_detection_elapsed is None:
            first_detection_elapsed = elapsed

        # Stage 2: DQN Action Selection
        t_dqn = time.perf_counter()
        obs    = build_dqn_obs(cal_prob, sample)
        action, _ = dqn_agent.predict(obs, deterministic=True)
        action = int(action)
        dqn_ms = (time.perf_counter() - t_dqn) * 1000.0

        target_ip   = "none"
        skip_reason = "not_nominated"
        raw_nominated_ip = "none"

        if action > 0:
            raw_nominated_ip = str(sample.get(f"src{action}_ip", "none")).strip()
            # ONLY skip sentinels (no hardcoded legit IPs)
            if raw_nominated_ip in _IP_SENTINELS:
                skip_reason = "sentinel_value"
            else:
                target_ip   = raw_nominated_ip
                skip_reason = "accepted"

        _ip_action_log.append({
            "step":              step,
            "elapsed_s":         elapsed,
            "gt_label":          gt_label,
            "attack_episode_id": attack_episode_id if in_attack else 0,
            "dqn_action":        action,
            "nominated_ip":      raw_nominated_ip,
            "target_ip":         target_ip,
            "skip_reason":       skip_reason,
            "is_sentinel_skip":  int(skip_reason == "sentinel_value"),
            "is_accepted":       int(skip_reason == "accepted"),
            "attacker_gt_ip":    gt_ip,
            "correct_target":    int(target_ip != "none" and target_ip == gt_ip),
        })

        mapped_action = 2 if (action > 0 and target_ip != "none") else 0

        # Mitigation Logic
        if gt_label == 1:
            mitigation.update(mapped_action, target_ip, step)
            if prob_at_mitigation_start is None and mitigation.active_action in (1, 2):
                prob_at_mitigation_start = cal_prob
                first_policy_block_elapsed = elapsed
            episode_probs.append(cal_prob)
        else:
            if mitigation.active_action != 0:
                mitigation.clear_all(step, reason="gt_is_0_cleanup")

        active_a = mitigation.active_action
        agent_overridden = (active_a != mapped_action) and (gt_label == 1)

        if active_a == 0:
            if gt_label == 0 and action > 0:
                action_str = f"FA_BLOCKED (ignored) src{action}({target_ip})"
            elif gt_label == 1 and action == 0:
                action_str = "SEARCHING..."
            else:
                action_str = "ALLOW"
        elif active_a == 1:
            action_str = f"RATE_LIMIT ({mitigation.active_ip})" if ACTIVE_MITIGATION else f"WOULD_RATE_LIMIT ({mitigation.active_ip})"
        elif active_a == 2:
            action_str = f"BLOCK ({mitigation.active_ip})" if ACTIVE_MITIGATION else f"WOULD_BLOCK ({mitigation.active_ip})"
        else:
            action_str = "UNKNOWN"

        mitigation_duration = (time.time() - mitigation.applied_at
                               if mitigation.applied_at else 0.0)

        now = time.time()
        if now - docker_cache_ts >= DOCKER_CACHE_SEC:
            docker_cache    = safe_get_docker_stats()
            docker_cache_ts = now

        onos1_cpu = float((docker_cache.get("onos1") or {}).get("cpu", 0))
        onos2_cpu = float((docker_cache.get("onos2") or {}).get("cpu", 0))
        onos3_cpu = float((docker_cache.get("onos3") or {}).get("cpu", 0))
        c_cpu_avg = (onos1_cpu + onos2_cpu + onos3_cpu) / 3.0
        c_cpu_max = max(onos1_cpu, onos2_cpu, onos3_cpu)

        py_cpu  = float(process.cpu_percent(interval=None))
        mem_mb  = process.memory_info().rss / (1024 * 1024)

        total_packets          = int(sample.get("total_packets", 0) or 0)
        total_packets_delta    = float(sample.get("total_packets_delta", total_packets) or 0.0)
        forwarded_packets_delta = float(sample.get("forwarded_packets_delta", total_packets_delta) or 0.0)
        dropped_packets_delta  = 0.0
        total_bytes            = int(sample.get("total_bytes", 0) or 0)
        total_flows            = int(sample.get("total_flows", 0) or 0)
        flow_arrival_rate      = float(sample.get("flow_arrival_rate", 0) or 0)
        src_entropy            = float(sample.get("src_entropy", sample.get("source_entropy", 0)) or 0)
        top_source_dominance   = float(sample.get("top_source_dominance", 0) or 0)
        active_sources         = int(sample.get("active_sources", 0) or 0)

        throughput_ratio = (forwarded_packets_delta / total_packets_delta
                            if total_packets_delta > 0 else 1.0)
        pkt_rate_pps = total_packets_delta / max(INTERVAL, 1)

        syn_ratio_top   = float(sample.get("src1_syn_ratio", sample.get("syn_ratio", 0)) or 0)
        pps_top_src     = float(sample.get("src1_pps", 0) or 0)

        prob_delta   = cal_prob - step_log[-1]["attack_prob"] if step_log else 0.0
        prob_slope_5 = 0.0
        if len(step_log) >= 5:
            recent5      = [s["attack_prob"] for s in step_log[-5:]]
            prob_slope_5 = (cal_prob - recent5[0]) / 5.0

        if step_log:
            prev = step_log[-1]
            cum_tp = prev["cum_tp"] + int(binary == 1 and gt_label == 1)
            cum_fp = prev["cum_fp"] + int(binary == 1 and gt_label == 0)
            cum_tn = prev["cum_tn"] + int(binary == 0 and gt_label == 0)
            cum_fn = prev["cum_fn"] + int(binary == 0 and gt_label == 1)
        else:
            cum_tp = int(binary == 1 and gt_label == 1)
            cum_fp = int(binary == 1 and gt_label == 0)
            cum_tn = int(binary == 0 and gt_label == 0)
            cum_fn = int(binary == 0 and gt_label == 1)

        roll_precision = cum_tp / max(cum_tp + cum_fp, 1)
        roll_recall    = cum_tp / max(cum_tp + cum_fn, 1)
        roll_f1        = (2 * roll_precision * roll_recall /
                          max(roll_precision + roll_recall, 1e-9))

        step_total_ms = (time.perf_counter() - step_t0) * 1000.0
        correct       = "OK" if binary == gt_label else "ERR"

        mit_tag      = "🔒" if active_a != 0 else "  "
        override_tag = f" (agent wanted {action})" if agent_overridden else ""
        backlog      = len(new_samples)
        lag_tag      = f" backlog={backlog}" if backlog > 1 else ""

        print(f"[{step:>4}] {mit_tag} Prob={cal_prob:.4f} (Δ={prob_delta:+.4f}) {correct} | "
              f"GT={gt_label}({gt_host}) | ⚡ {action_str}{override_tag} | "
              f"ctrl_cpu={c_cpu_avg:.1f}% | mit_dur={mitigation_duration:.0f}s "
              f"[{step_total_ms:.0f}ms]{lag_tag}")

        step_log.append({
            "step":                     step,
            "sample_id":                sample_id,
            "elapsed_s":                elapsed,
            "timestamp":                datetime.now().isoformat(),
            "backlog_samples":          backlog,
            "gt_label":                 gt_label,
            "gt_host":                  gt_host,
            "gt_ip":                    gt_ip,
            "gt_source":                gt_source,
            "gt_file_label":            gt_file_label,
            "gt_sample_label":          gt_sample_label,
            "gt_mismatch":              gt_mismatch,
            "attack_episode_id":        attack_episode_id if in_attack else 0,
            "raw_prob":                 round(raw_prob, 6),
            "attack_prob":              round(cal_prob, 6),
            "binary_pred":              binary,
            "threshold":                BEST_THRESHOLD,
            "correct":                  int(binary == gt_label),
            "is_tp":                    int(binary == 1 and gt_label == 1),
            "is_fp":                    int(binary == 1 and gt_label == 0),
            "is_tn":                    int(binary == 0 and gt_label == 0),
            "is_fn":                    int(binary == 0 and gt_label == 1),
            "prob_delta":               round(prob_delta, 6),
            "prob_slope_5step":         round(prob_slope_5, 6),
            "prob_at_mitigation_start": prob_at_mitigation_start,
            "prob_change_since_mitig":  (round(cal_prob - prob_at_mitigation_start, 6)
                                         if prob_at_mitigation_start is not None else None),
            "cum_tp":                   cum_tp,
            "cum_fp":                   cum_fp,
            "cum_tn":                   cum_tn,
            "cum_fn":                   cum_fn,
            "rolling_precision":        round(roll_precision, 4),
            "rolling_recall":           round(roll_recall, 4),
            "rolling_f1":               round(roll_f1, 4),
            "dqn_raw_action":           action,
            "mapped_action":            mapped_action,
            "active_mitigation":        active_a,
            "agent_overridden":         agent_overridden,
            "action_taken":             action_str,
            "nominated_ip":             raw_nominated_ip,
            "skip_reason":              skip_reason,
            "target_ip":                target_ip,
            "would_mitigate":           int(active_a > 0),
            "would_mitigated_ip":       mitigation.active_ip if active_a > 0 else "",
            "mitigation_active":        int(active_a > 0),
            "mitigation_duration_s":    round(mitigation_duration, 2),
            "total_packets":            total_packets,
            "total_packets_delta":      total_packets_delta,
            "forwarded_packets_delta":  forwarded_packets_delta,
            "dropped_packets_delta":    dropped_packets_delta,
            "throughput_ratio":         round(throughput_ratio, 4),
            "pkt_rate_pps":             round(pkt_rate_pps, 2),
            "total_flows":              total_flows,
            "total_bytes":              total_bytes,
            "flow_arrival_rate":        round(flow_arrival_rate, 4),
            "src_entropy":              round(src_entropy, 4),
            "top_source_dominance":     round(top_source_dominance, 4),
            "active_sources":           active_sources,
            "syn_ratio_top_src":        round(syn_ratio_top, 4),
            "pps_top_src":              round(pps_top_src, 2),
            "attack_start_elapsed":         attack_start_elapsed,
            "first_detection_elapsed":      first_detection_elapsed,
            "first_policy_block_elapsed":   first_policy_block_elapsed,
            "onos1_cpu":                round(onos1_cpu, 2),
            "onos2_cpu":                round(onos2_cpu, 2),
            "onos3_cpu":                round(onos3_cpu, 2),
            "ctrl_cpu_avg":             round(c_cpu_avg, 2),
            "ctrl_cpu_max":             round(c_cpu_max, 2),
            "python_cpu_pct":           round(py_cpu, 2),
            "python_mem_mb":            round(mem_mb, 1),
            "preprocess_ms":            round(preprocess_ms, 2),
            "lstm_inference_ms":        round(lstm_ms, 2),
            "dqn_inference_ms":         round(dqn_ms, 2),
            "api_latency_ms":           0.0,
            "drop_query_ms":            0.0,
            "step_total_ms":            round(step_total_ms, 2),
        })
        step += 1

except KeyboardInterrupt:
    pass

if mitigation.active_action != 0:
    mitigation.clear_all(step, reason="engine_shutdown")
    print("[ENGINE] Cleaned up would-be mitigation state on shutdown.")
else:
    mitigation._clear_drop_rules()
    mitigation._remove_rate_limit()

elapsed_total = time.time() - t_start
print(f"\n[ENGINE] Total: {step} steps in {elapsed_total:.0f}s")

if not step_log:
    print("[ENGINE] No steps logged — nothing to save.")
    raise SystemExit(0)

# =====================================================================
# 📊 BUILD ALL DATAFRAMES
# =====================================================================
df_log = pd.DataFrame(step_log)

episodes = []
for eid in df_log["attack_episode_id"].unique():
    if eid == 0: continue
    ep     = df_log[df_log["attack_episode_id"] == eid]
    ep_mit = ep[ep["active_mitigation"] > 0]
    ep_blk = ep[ep["active_mitigation"] == 2]
    ep_rl  = ep[ep["active_mitigation"] == 1]

    det_lat_val = None
    pol_lat_val = None
    ep_start_el = ep["attack_start_elapsed"].dropna()
    ep_det_el   = ep["first_detection_elapsed"].dropna()
    ep_pol_el   = ep["first_policy_block_elapsed"].dropna()
    if len(ep_start_el) and len(ep_det_el):
        det_lat_val = round(ep_det_el.iloc[0] - ep_start_el.iloc[0], 2)
    if len(ep_start_el) and len(ep_pol_el):
        pol_lat_val = round(ep_pol_el.iloc[0] - ep_start_el.iloc[0], 2)

    episodes.append({
        "episode_id":               eid,
        "start_step":               int(ep["step"].iloc[0]),
        "end_step":                 int(ep["step"].iloc[-1]),
        "duration_steps":           len(ep),
        "duration_seconds":         round(ep["elapsed_s"].iloc[-1] - ep["elapsed_s"].iloc[0], 1),
        "attacker_host":            ep["gt_host"].iloc[0],
        "attacker_ip":              ep["gt_ip"].iloc[0],
        "prob_mean":                round(ep["attack_prob"].mean(), 4),
        "prob_max":                 round(ep["attack_prob"].max(), 4),
        "prob_min":                 round(ep["attack_prob"].min(), 4),
        "prob_std":                 round(ep["attack_prob"].std(), 4),
        "prob_at_start":            round(ep["attack_prob"].iloc[0], 4),
        "prob_at_end":              round(ep["attack_prob"].iloc[-1], 4),
        "steps_above_threshold":    int((ep["attack_prob"] >= BEST_THRESHOLD).sum()),
        "detection_rate":           round((ep["attack_prob"] >= BEST_THRESHOLD).mean(), 4),
        "detection_latency_s":      det_lat_val,
        "policy_reaction_latency_s": pol_lat_val,
        "episode_tp":               int(ep["is_tp"].sum()),
        "episode_fp":               int(ep["is_fp"].sum()),
        "episode_tn":               int(ep["is_tn"].sum()),
        "episode_fn":               int(ep["is_fn"].sum()),
        "episode_precision":        round(ep["is_tp"].sum() / max(ep["is_tp"].sum() + ep["is_fp"].sum(), 1), 4),
        "episode_recall":           round(ep["is_tp"].sum() / max(ep["is_tp"].sum() + ep["is_fn"].sum(), 1), 4),
        "steps_would_mitigate":     len(ep_mit),
        "mitigation_coverage":      round(len(ep_mit) / len(ep), 4) if len(ep) > 0 else 0,
        "would_block_steps":        len(ep_blk),
        "would_rate_limit_steps":   len(ep_rl),
        "allow_steps":              int((ep["active_mitigation"] == 0).sum()),
        "unique_target_ips":        ep[ep["target_ip"] != "none"]["target_ip"].nunique(),
        "avg_total_packets":        round(ep["total_packets"].mean(), 1),
        "avg_pkt_rate_pps":         round(ep["pkt_rate_pps"].mean(), 2),
        "avg_fwd_packets_delta":    round(ep["forwarded_packets_delta"].mean(), 1),
        "avg_throughput_ratio":     round(ep["throughput_ratio"].mean(), 4),
        "avg_src_entropy":          round(ep["src_entropy"].mean(), 4),
        "avg_top_dominance":        round(ep["top_source_dominance"].mean(), 4),
        "avg_syn_ratio_top":        round(ep["syn_ratio_top_src"].mean(), 4),
        "ctrl_cpu_mean":            round(ep["ctrl_cpu_avg"].mean(), 2),
        "ctrl_cpu_peak":            round(ep["ctrl_cpu_max"].max(), 2),
        "engine_cpu_mean":          round(ep["python_cpu_pct"].mean(), 2),
        "lstm_ms_mean":             round(ep["lstm_inference_ms"].mean(), 2),
        "dqn_ms_mean":              round(ep["dqn_inference_ms"].mean(), 2),
        "step_total_ms_mean":       round(ep["step_total_ms"].mean(), 2),
    })

df_ep = pd.DataFrame(episodes) if episodes else pd.DataFrame()

df_trans = pd.DataFrame(mitigation.history, columns=["step", "old_action", "new_action", "target_ip", "reason"]) if mitigation.history else pd.DataFrame(columns=["step", "old_action", "new_action", "target_ip", "reason"])
action_map = {0: "ALLOW", 1: "RATE_LIMIT", 2: "BLOCK"}
df_trans["old_action_name"] = df_trans["old_action"].map(action_map).fillna("UNKNOWN")
df_trans["new_action_name"] = df_trans["new_action"].map(action_map).fillna("UNKNOWN")

df_detection = df_log[["step", "elapsed_s", "gt_label", "attack_episode_id", "raw_prob", "attack_prob", "binary_pred", "is_tp", "is_fp", "is_tn", "is_fn", "cum_tp", "cum_fp", "cum_tn", "cum_fn", "rolling_precision", "rolling_recall", "rolling_f1"]].copy()

f1  = f1_score(df_log["gt_label"], df_log["binary_pred"], zero_division=0)
pre = precision_score(df_log["gt_label"], df_log["binary_pred"], zero_division=0)
rec = recall_score(df_log["gt_label"], df_log["binary_pred"], zero_division=0)
cm  = confusion_matrix(df_log["gt_label"], df_log["binary_pred"], labels=[0, 1])
tn_total, fp_total, fn_total, tp_total = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
try: roc_auc = roc_auc_score(df_log["gt_label"], df_log["attack_prob"])
except: roc_auc = float("nan")
try: pr_auc = average_precision_score(df_log["gt_label"], df_log["attack_prob"])
except: pr_auc = float("nan")

df_det_summary = pd.DataFrame([
    {"metric": "Total Steps",          "value": step},
    {"metric": "Attack Steps",        "value": int((df_log["gt_label"] == 1).sum())},
    {"metric": "Benign Steps",        "value": int((df_log["gt_label"] == 0).sum())},
    {"metric": "TP",                  "value": tp_total},
    {"metric": "FP",                  "value": fp_total},
    {"metric": "TN",                  "value": tn_total},
    {"metric": "FN",                  "value": fn_total},
    {"metric": "Precision",           "value": round(pre, 4)},
    {"metric": "Recall",              "value": round(rec, 4)},
    {"metric": "F1 Score",            "value": round(f1, 4)},
    {"metric": "ROC AUC",             "value": round(roc_auc, 4) if not np.isnan(roc_auc) else "N/A"},
    {"metric": "PR AUC",              "value": round(pr_auc, 4) if not np.isnan(pr_auc) else "N/A"},
    {"metric": "Attack Episodes",     "value": int(df_log["attack_episode_id"].max())},
    {"metric": "Would-Mitigate Steps","value": int((df_log["active_mitigation"] > 0).sum())},
    {"metric": "FP Mitigations",      "value": int(((df_log["gt_label"] == 0) & (df_log["active_mitigation"] > 0)).sum())},
    {"metric": "LSTM Threshold",      "value": BEST_THRESHOLD},
    {"metric": "Temperature",         "value": TEMPERATURE},
    {"metric": "Window Length",       "value": WINDOW_LEN},
])

lat_cols = ["preprocess_ms", "lstm_inference_ms", "dqn_inference_ms", "step_total_ms"]
lat_rows = []
for c in lat_cols:
    arr = df_log[c].dropna().values
    lat_rows.append({
        "component": c, "mean_ms": round(arr.mean(), 3), "median_ms": round(np.median(arr), 3),
        "p95_ms": round(np.percentile(arr, 95), 3), "p99_ms": round(np.percentile(arr, 99), 3),
        "max_ms": round(arr.max(), 3), "min_ms": round(arr.min(), 3), "std_ms": round(arr.std(), 3), "samples": len(arr),
    })
for phase, mask in [("benign", df_log["gt_label"] == 0), ("attack", df_log["gt_label"] == 1)]:
    arr = df_log.loc[mask, "lstm_inference_ms"].dropna().values
    if len(arr):
        lat_rows.append({
            "component": f"lstm_inference_ms [{phase}]", "mean_ms": round(arr.mean(), 3), "median_ms": round(np.median(arr), 3),
            "p95_ms": round(np.percentile(arr, 95), 3), "p99_ms": round(np.percentile(arr, 99), 3),
            "max_ms": round(arr.max(), 3), "min_ms": round(arr.min(), 3), "std_ms": round(arr.std(), 3), "samples": len(arr),
        })

df_latency = pd.DataFrame(lat_rows)

traffic_cols = ["elapsed_s", "gt_label", "attack_episode_id", "total_packets", "total_packets_delta", "forwarded_packets_delta", "dropped_packets_delta", "throughput_ratio", "pkt_rate_pps", "total_flows", "total_bytes", "flow_arrival_rate", "src_entropy", "top_source_dominance", "active_sources", "syn_ratio_top_src", "pps_top_src", "ctrl_cpu_avg", "ctrl_cpu_max", "python_cpu_pct", "python_mem_mb"]
df_traffic = df_log[traffic_cols].copy()

df_ip_raw = pd.DataFrame(_ip_action_log) if _ip_action_log else pd.DataFrame(
    columns=["step", "elapsed_s", "gt_label", "attack_episode_id", "dqn_action", "nominated_ip", "target_ip", "skip_reason", "is_sentinel_skip", "is_accepted", "attacker_gt_ip", "correct_target"])

ip_summary_rows = []
if not df_ip_raw.empty:
    for ip, grp in df_ip_raw[df_ip_raw["nominated_ip"] != "none"].groupby("nominated_ip"):
        grp_atk  = grp[grp["gt_label"] == 1]
        grp_ben  = grp[grp["gt_label"] == 0]
        ip_summary_rows.append({
            "nominated_ip":         ip,
            "classification":       ("sentinel" if ip in _IP_SENTINELS else "unknown_accepted"),
            "total_nominations":    len(grp),
            "nominations_during_attack":  len(grp_atk),
            "nominations_during_benign":  len(grp_ben),
            "times_accepted":       int(grp["is_accepted"].sum()),
            "times_sentinel_skipped": int(grp["is_sentinel_skip"].sum()),
            "correct_target_hits":  int(grp["correct_target"].sum()),
            "correct_target_rate":  round(grp["correct_target"].sum() / max(grp["is_accepted"].sum(), 1), 4),
            "first_seen_step":      int(grp["step"].min()),
            "last_seen_step":       int(grp["step"].max()),
            "skip_reason_sample":   grp["skip_reason"].iloc[0],
        })

df_ip_summary = pd.DataFrame(ip_summary_rows) if ip_summary_rows else pd.DataFrame(columns=["nominated_ip", "classification", "total_nominations", "nominations_during_attack", "nominations_during_benign", "times_accepted", "times_sentinel_skipped", "correct_target_hits", "correct_target_rate", "first_seen_step", "last_seen_step", "skip_reason_sample"])

try:
    import platform, socket
    hostname = socket.gethostname(); os_info = platform.platform(); python_ver = platform.python_version()
except: hostname = os_info = python_ver = "unknown"

df_meta = pd.DataFrame([
    {"key": "run_start_time", "value": RUN_START_TIME}, {"key": "total_duration_s", "value": round(elapsed_total, 1)},
    {"key": "total_steps", "value": step}, {"key": "active_mitigation", "value": str(ACTIVE_MITIGATION)},
    {"key": "run_mode", "value": _RUN_MODE}, {"key": "ip_sentinels_always_skipped", "value": ", ".join(sorted(_IP_SENTINELS))}
])

try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False

csv_map = {
    "live_drl_log_nomit.csv": df_log, "episode_summary.csv": df_ep, "mitigation_transitions.csv": df_trans,
    "detection_stats_per_step.csv": df_detection, "detection_summary.csv": df_det_summary,
    "latency_profile.csv": df_latency, "traffic_analysis.csv": df_traffic,
    "source_ip_summary.csv": df_ip_summary, "source_ip_raw_log.csv": df_ip_raw, "run_metadata.csv": df_meta,
}
for fname, df in csv_map.items(): df.to_csv(SAVE_DIR / fname, index=False)

if HAS_OPENPYXL:
    wb = Workbook(); wb.remove(wb.active)
    
    def write_df_to_sheet(ws, df, start_row=1):
        for ci, col in enumerate(df.columns, 1): ws.cell(row=start_row, column=ci, value=col)
        for ri, (_, row) in enumerate(df.iterrows(), start_row + 1):
            for ci, col in enumerate(df.columns, 1): ws.cell(row=ri, column=ci, value=row[col])
        return start_row + len(df) + 1

    ws1 = wb.create_sheet("Step Log"); write_df_to_sheet(ws1, df_log)
    ws2 = wb.create_sheet("Episode Summary"); write_df_to_sheet(ws2, df_ep) if not df_ep.empty else None
    ws3 = wb.create_sheet("Transitions"); write_df_to_sheet(ws3, df_trans) if not df_trans.empty else None
    ws4 = wb.create_sheet("Detection Stats"); write_df_to_sheet(ws4, df_det_summary)
    ws5 = wb.create_sheet("Latency Profile"); write_df_to_sheet(ws5, df_latency)
    ws6 = wb.create_sheet("Traffic Analysis"); write_df_to_sheet(ws6, df_traffic)
    ws7 = wb.create_sheet("Source IP Analysis"); write_df_to_sheet(ws7, df_ip_summary)
    ws8 = wb.create_sheet("Run Metadata"); write_df_to_sheet(ws8, df_meta)
    
    xlsx_path = SAVE_DIR / f"live_drl_{_RUN_MODE}_results.xlsx"
    wb.save(str(xlsx_path))
    print(f"\n[SAVED] Excel workbook (8 sheets) → {xlsx_path}")


[ENGINE] Output folder: /home/shokran/PycharmProjects/JupyterProject/Models/newTrail/outputs_drl_active_mitigation
[ENGINE] Stage 1 (LSTM) Ready. Window=5, Thresh=0.8472, T=0.9568
[ENGINE] Stage 2 (DQN) Action Space: 0=ALLOW, 1..6=BLOCK_SRC
[ENGINE] Switches: ['of:0000000000000001', 'of:0000000000000002', 'of:0000000000000003', 'of:0000000000000004']
[ENGINE] ⚠️  ACTIVE_MITIGATION=True

🚀 LIVE DQN MODE (Active Mitigation: True)

[   0]    Prob=0.0020 (Δ=+0.0000) OK | GT=0(none) | ⚡ ALLOW | ctrl_cpu=3.1% | mit_dur=0s [1584ms] backlog=205
[   1]    Prob=0.0029 (Δ=+0.0009) OK | GT=0(none) | ⚡ ALLOW | ctrl_cpu=4.9% | mit_dur=0s [2065ms]
[   2]    Prob=0.0023 (Δ=-0.0006) OK | GT=0(none) | ⚡ ALLOW | ctrl_cpu=4.7% | mit_dur=0s [2084ms] backlog=2
[   3]    Prob=0.0020 (Δ=-0.0003) OK | GT=0(none) | ⚡ ALLOW | ctrl_cpu=4.1% | mit_dur=0s [2051ms] backlog=2
[   4]    Prob=0.0021 (Δ=+0.0001) OK | GT=0(none) | ⚡ ALLOW | ctrl_cpu=5.2% | mit_dur=0s [2057ms] backlog=2
[   5]    Prob=0.0021 (Δ=-0.0000) O

In [43]:
stop_monitoring()

✓ Monitoring stopped. 458 samples saved to: DRL_test_1.csv
